In [ ]:
Data Preprocessing

In [ ]:
# ============================================================
# Key design:
#   - split UDS into modalities FIRST
#   - DO NOT one-hot encode
#   - each original variable stays as exactly one column
#   - categorical / ordinal variables are converted to one numeric code each
#   - numeric variables are median-imputed and standardized
#   - categorical / ordinal variables are mode-imputed
#   - MRI is matched to each UDS visit within 18 months
#   - unmatched MRI rows are kept and filled with MISSING_VALUE
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# ============================================================
# PATHS
# ============================================================

RAW_DIR = "./raw_data"
OUT_DIR = "./preprocessed_v3_for_joint_missingMRI_keepRawDims"
os.makedirs(OUT_DIR, exist_ok=True)

uds_path = os.path.join(RAW_DIR, "investigator_nacc64.csv")
mrisbm_path = os.path.join(
    RAW_DIR,
    "investigator_scan_mri_nacc66",
    "investigator_scan_mrisbm_nacc66.csv"
)

MISSING_VALUE = -999.0
MRI_WINDOW_DAYS = 18 * 30  # approximate 18 months


# ============================================================
# FEATURE DEFINITIONS
# ============================================================

def get_feature_types(df, tier2=True):
    cat_demo_feats = ['NACCNIHR','PRIMLANG','SEX','HISPANIC']
    ord_demo_feats = ['MARISTAT','NACCLIVS','INDEPEND','RESIDENC']
    num_demo_feats = ['NACCAGE','EDUC']

    cat_phist_feats = ['TOBAC30', 'TOBAC100','NACCTBI','DEP2YRS', 'DEPOTHR',
            'ANYMEDS','NACCAAAS', 'NACCAANX', 'NACCAC', 'NACCACEI',
            'NACCADEP', 'NACCAHTN', 'NACCANGI', 'NACCAPSY',
            'NACCBETA', 'NACCCCBS', 'NACCDBMD', 'NACCDIUR', 'NACCEMD',
            'NACCEPMD', 'NACCHTNC', 'NACCLIPL', 'NACCNSD', 'NACCPDMD',
            'NACCVASD']
    ord_phist_feats = ['NACCAMD','PACKSPER','CVHATT', 'CVAFIB', 'CVANGIO', 'CVBYPASS', 'CBTIA',
            'CVPACE', 'CVCHF', 'CVOTHR', 'CBSTROKE','SEIZURES','NCOTHR',
            'DIABETES','HYPERTEN', 'HYPERCHO', 'B12DEF','THYROID', 'INCONTU',
            'INCONTF','ALCOHOL', 'ABUSOTHR','PSYCDIS']
    num_phist_feats = ['SMOKYRS','NACCSTYR','NACCTIYR']

    cat_fhist_feats = ['NACCFADM', 'NACCFFTD']
    ord_fhist_feats = ['NACCFAM', 'NACCMOM', 'NACCDAD']

    cat_phys_feats = ['NACCNREX','FOCLSYM','FOCLSIGN']
    ord_phys_feats = ['DECSUB','VISION', 'VISCORR','VISWCORR','HEARING', 'HEARAID', 'HEARWAID']
    num_phys_feats = ['HEIGHT', 'WEIGHT','BPSYS', 'BPDIAS', 'HRATE','NACCBMI']

    ord_gds_feats = ['NOGDS','SATIS', 'DROPACT', 'EMPTY', 'BORED', 'SPIRITS', 'AFRAID',
            'HAPPY', 'HELPLESS', 'STAYHOME', 'MEMPROB', 'WONDRFUL', 'WRTHLESS',
            'ENERGY', 'HOPELESS', 'BETTER','NACCGDS']
    ord_faq_feats = ['BILLS', 'TAXES','SHOPPING', 'GAMES', 'STOVE',
            'MEALPREP', 'EVENTS', 'PAYATTN','REMDATES', 'TRAVEL']
    ord_npi_feats = ['DELSEV', 'HALLSEV', 'AGITSEV', 'DEPDSEV', 'ANXSEV',
                'ELATSEV', 'APASEV', 'DISNSEV', 'IRRSEV', 'MOTSEV', 'NITESEV',
                'APPSEV']

    ord_np_feats = ['MMSEORDA','MMSEORLO']

    label_feat = ['NACCUDSD']
    cdr_feats = ['MEMORY', 'ORIENT', 'JUDGMENT', 'COMMUN', 'HOMEHOBB',
                 'PERSCARE', 'CDRSUM', 'CDRGLOB']

    cat_feats = cat_demo_feats + cat_phist_feats + cat_fhist_feats + cat_phys_feats

    if tier2:
        num_np_feats = ['NACCMMSE','MEMUNITS','DIGIF', 'DIGIFLEN', 'DIGIB', 'DIGIBLEN',
                        'ANIMALS', 'VEG','BOSTON', 'TRAILA', 'TRAILB']
    else:
        num_np_feats = ['NACCMMSE']

    ord_feats = list(ord_demo_feats + ord_phist_feats + ord_fhist_feats + ord_phys_feats +
                     ord_npi_feats + ord_gds_feats + ord_faq_feats + ord_np_feats)
    num_feats = num_demo_feats + num_phist_feats + num_phys_feats + num_np_feats

    cat_feats = [x for x in cat_feats if x in df.columns]
    ord_feats = [x for x in ord_feats if x in df.columns]
    num_feats = [x for x in num_feats if x in df.columns]
    label_feat = [x for x in label_feat if x in df.columns]
    cdr_feats = [x for x in cdr_feats if x in df.columns]

    return cat_feats, ord_feats, num_feats, label_feat, cdr_feats


def get_modality_features(df, tier2=True, fine_grained=False):
    cat_demo_feats = ['NACCNIHR','PRIMLANG','SEX','HISPANIC']
    ord_demo_feats = ['MARISTAT','NACCLIVS','INDEPEND','RESIDENC']
    num_demo_feats = ['NACCAGE','EDUC']

    cat_phist_feats = ['TOBAC30', 'TOBAC100','NACCTBI','DEP2YRS', 'DEPOTHR',
            'ANYMEDS','NACCAAAS', 'NACCAANX', 'NACCAC', 'NACCACEI',
            'NACCADEP', 'NACCAHTN', 'NACCANGI', 'NACCAPSY',
            'NACCBETA', 'NACCCCBS', 'NACCDBMD', 'NACCDIUR', 'NACCEMD',
            'NACCEPMD', 'NACCHTNC', 'NACCLIPL', 'NACCNSD', 'NACCPDMD',
            'NACCVASD']
    ord_phist_feats = ['NACCAMD','PACKSPER','CVHATT', 'CVAFIB', 'CVANGIO', 'CVBYPASS', 'CBTIA',
            'CVPACE', 'CVCHF', 'CVOTHR', 'CBSTROKE','SEIZURES','NCOTHR',
            'DIABETES','HYPERTEN', 'HYPERCHO', 'B12DEF','THYROID', 'INCONTU',
            'INCONTF','ALCOHOL', 'ABUSOTHR','PSYCDIS']
    num_phist_feats = ['SMOKYRS','NACCSTYR','NACCTIYR']

    cat_fhist_feats = ['NACCFADM', 'NACCFFTD']
    ord_fhist_feats = ['NACCFAM', 'NACCMOM', 'NACCDAD']

    cat_phys_feats = ['NACCNREX','FOCLSYM','FOCLSIGN']
    ord_phys_feats = ['DECSUB','VISION', 'VISCORR','VISWCORR','HEARING', 'HEARAID', 'HEARWAID']
    num_phys_feats = ['HEIGHT', 'WEIGHT','BPSYS', 'BPDIAS', 'HRATE','NACCBMI']

    ord_gds_feats = ['NOGDS','SATIS', 'DROPACT', 'EMPTY', 'BORED', 'SPIRITS', 'AFRAID',
            'HAPPY', 'HELPLESS', 'STAYHOME', 'MEMPROB', 'WONDRFUL', 'WRTHLESS',
            'ENERGY', 'HOPELESS', 'BETTER','NACCGDS']
    ord_faq_feats = ['BILLS', 'TAXES','SHOPPING', 'GAMES', 'STOVE',
            'MEALPREP', 'EVENTS', 'PAYATTN','REMDATES', 'TRAVEL']
    ord_npi_feats = ['DELSEV', 'HALLSEV', 'AGITSEV', 'DEPDSEV', 'ANXSEV',
                'ELATSEV', 'APASEV', 'DISNSEV', 'IRRSEV', 'MOTSEV', 'NITESEV',
                'APPSEV']

    ord_np_feats = ['MMSEORDA','MMSEORLO']
    if tier2:
        num_np_feats = ['NACCMMSE','MEMUNITS','DIGIF', 'DIGIFLEN', 'DIGIB', 'DIGIBLEN',
                        'ANIMALS', 'VEG','BOSTON', 'TRAILA', 'TRAILB']
    else:
        num_np_feats = ['NACCMMSE']

    label_feat = ['NACCUDSD']

    demo_feats = cat_demo_feats + ord_demo_feats + num_demo_feats
    phist_feats = cat_phist_feats + ord_phist_feats + num_phist_feats
    fhist_feats = cat_fhist_feats + ord_fhist_feats

    mod_history = [x for x in (demo_feats + phist_feats + fhist_feats) if x in df.columns]

    phys_feats = cat_phys_feats + ord_phys_feats + num_phys_feats
    npi_feats = ord_gds_feats + ord_faq_feats + ord_npi_feats
    npt_feats = ord_np_feats + num_np_feats

    if fine_grained:
        mod_survey = [x for x in npi_feats if x in df.columns]
        mod_testing = [x for x in (phys_feats + npt_feats) if x in df.columns]
        return mod_history, mod_survey, mod_testing, label_feat
    else:
        mod_survey = [x for x in (phys_feats + npi_feats + npt_feats) if x in df.columns]
        return mod_history, mod_survey, label_feat


# ============================================================
# MODALITY PREPROCESSING WITHOUT ONE-HOT
# ============================================================

def preprocess_one_modality_keep_width(df_modality):
    """
    Keeps exactly one output column per original input column.
    No one-hot encoding.
    """
    x = df_modality.copy()

    cat_feats, ord_feats, num_feats, _, _ = get_feature_types(x, tier2=True)
    cat_feats = [c for c in cat_feats if c in x.columns]
    ord_feats = [c for c in ord_feats if c in x.columns]
    num_feats = [c for c in num_feats if c in x.columns]

    out = pd.DataFrame(index=x.index)

    # categorical: mode impute + ordinal encode to ONE column each
    if len(cat_feats) > 0:
        x_cat = x[cat_feats].copy()
        cat_imputer = SimpleImputer(strategy='most_frequent')
        x_cat_imp = pd.DataFrame(cat_imputer.fit_transform(x_cat), columns=cat_feats, index=x.index)

        cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        x_cat_enc = pd.DataFrame(cat_encoder.fit_transform(x_cat_imp), columns=cat_feats, index=x.index)
        out = pd.concat([out, x_cat_enc], axis=1)

    # ordinal: mode impute, keep one column each
    if len(ord_feats) > 0:
        x_ord = x[ord_feats].copy()
        for c in ord_feats:
            x_ord[c] = pd.to_numeric(x_ord[c], errors='coerce')
        ord_imputer = SimpleImputer(strategy='most_frequent')
        x_ord_imp = pd.DataFrame(ord_imputer.fit_transform(x_ord), columns=ord_feats, index=x.index)
        out = pd.concat([out, x_ord_imp], axis=1)

    # numeric: median impute + z-score, one column each
    if len(num_feats) > 0:
        x_num = x[num_feats].copy()
        for c in num_feats:
            x_num[c] = pd.to_numeric(x_num[c], errors='coerce')
        num_imputer = SimpleImputer(strategy='median')
        x_num_imp = pd.DataFrame(num_imputer.fit_transform(x_num), columns=num_feats, index=x.index)

        scaler = StandardScaler()
        x_num_scl = pd.DataFrame(scaler.fit_transform(x_num_imp), columns=num_feats, index=x.index)
        out = pd.concat([out, x_num_scl], axis=1)

    # preserve original column order
    out = out[[c for c in x.columns if c in out.columns]].astype(np.float32)
    return out


def preprocess_mri_full(df_mri):
    x = df_mri.copy()
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors='coerce')

    for c in x.columns:
        med = x[c].median()
        if pd.isna(med):
            med = 0.0
        x[c] = x[c].fillna(med)

    mu = x.mean(axis=0)
    sd = x.std(axis=0).replace(0, 1.0)
    x = ((x - mu) / sd).astype(np.float32)
    return x


def build_label_from_naccudsd(series):
    s = pd.to_numeric(series, errors='coerce')
    keep = s.isin([1, 2, 3, 4])

    out = pd.Series(index=series.index, dtype='float')
    out.loc[keep & s.isin([1, 2])] = 0
    out.loc[keep & (s == 3)] = 1
    out.loc[keep & (s == 4)] = 2
    return out, keep


def choose_id_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of these ID columns were found: {candidates}")


def build_uds_datetime(df):
    if all(c in df.columns for c in ["VISITYR", "VISITMO", "VISITDAY"]):
        yy = pd.to_numeric(df["VISITYR"], errors="coerce")
        mm = pd.to_numeric(df["VISITMO"], errors="coerce")
        dd = pd.to_numeric(df["VISITDAY"], errors="coerce")
        return pd.to_datetime(dict(year=yy, month=mm, day=dd), errors="coerce")
    raise ValueError("Could not build UDS date; expected VISITYR/VISITMO/VISITDAY.")


def find_best_mri_match_for_each_uds(uds_df, mri_df, id_col, uds_date_series, mri_date_col, max_days=MRI_WINDOW_DAYS):
    mri_dates = pd.to_datetime(mri_df[mri_date_col], errors='coerce')
    mri_df_local = mri_df.copy()
    mri_df_local["_MRI_DATE_"] = mri_dates

    match_idx = []

    for i, row in uds_df.iterrows():
        sid = row[id_col]
        uds_date = uds_date_series.loc[i]

        cand = mri_df_local[mri_df_local[id_col] == sid].copy()
        if len(cand) == 0 or pd.isna(uds_date):
            match_idx.append(np.nan)
            continue

        cand["_ABS_DAYS_"] = (cand["_MRI_DATE_"] - uds_date).abs().dt.days
        cand = cand[cand["_ABS_DAYS_"] <= max_days]

        if len(cand) == 0:
            match_idx.append(np.nan)
            continue

        best = cand.sort_values("_ABS_DAYS_").index[0]
        match_idx.append(best)

    return pd.Series(match_idx, index=uds_df.index)


# ============================================================
# LOAD RAW DATA
# ============================================================

uds_raw = pd.read_csv(uds_path, low_memory=False)
mri_raw = pd.read_csv(mrisbm_path, low_memory=False)

print("Raw UDS shape :", uds_raw.shape)
print("Raw MRI shape :", mri_raw.shape)

ID_CANDIDATES = ["NACCID", "RID", "ID"]
MRI_DATE_CANDIDATES = ["SCANDT", "MRIDATE", "EXAMDATE"]

id_col = choose_id_column(uds_raw, ID_CANDIDATES)
mri_id_col = choose_id_column(mri_raw, ID_CANDIDATES)
if mri_id_col != id_col:
    mri_raw = mri_raw.rename(columns={mri_id_col: id_col})

mri_date_col = None
for c in MRI_DATE_CANDIDATES:
    if c in mri_raw.columns:
        mri_date_col = c
        break
if mri_date_col is None:
    raise ValueError(f"Could not find MRI date column among {MRI_DATE_CANDIDATES}")

uds_visit_date = build_uds_datetime(uds_raw)

print("Using ID column    :", id_col)
print("Using MRI date col :", mri_date_col)


# ============================================================
# LABEL + ELIGIBLE UDS VISITS
# ============================================================

diag_label, keep_diag = build_label_from_naccudsd(uds_raw["NACCUDSD"])
uds_keep = uds_raw.loc[keep_diag].copy().reset_index(drop=True)
uds_keep["LABEL"] = diag_label.loc[keep_diag].astype(int).values
uds_keep["_UDS_DATE_"] = uds_visit_date.loc[keep_diag].reset_index(drop=True)

print("Eligible UDS visits after diagnosis filtering:", uds_keep.shape[0])


# ============================================================
# SPLIT RAW UDS INTO 3 RAW MODALITIES FIRST
# ============================================================

mod_history, mod_survey, mod_testing, _ = get_modality_features(uds_keep, fine_grained=True)

uds_history_raw = uds_keep[mod_history].copy()
uds_survey_raw = uds_keep[mod_survey].copy()
uds_testing_raw = uds_keep[mod_testing].copy()

print("Raw modality dimensions before preprocessing:")
print("  history :", uds_history_raw.shape)
print("  survey  :", uds_survey_raw.shape)
print("  testing :", uds_testing_raw.shape)

uds_history = preprocess_one_modality_keep_width(uds_history_raw)
uds_survey = preprocess_one_modality_keep_width(uds_survey_raw)
uds_testing = preprocess_one_modality_keep_width(uds_testing_raw)

print("\nProcessed modality dimensions after KEEP-WIDTH preprocessing:")
print("  history :", uds_history.shape)
print("  survey  :", uds_survey.shape)
print("  testing :", uds_testing.shape)


# ============================================================
# MRI MATCHING AT UDS-VISIT LEVEL
# ============================================================

uds_meta = uds_keep[[id_col, "LABEL", "_UDS_DATE_"]].copy()

match_idx = find_best_mri_match_for_each_uds(
    uds_df=uds_meta,
    mri_df=mri_raw,
    id_col=id_col,
    uds_date_series=uds_meta["_UDS_DATE_"],
    mri_date_col=mri_date_col,
    max_days=MRI_WINDOW_DAYS
)

mri_available_mask = match_idx.notna().astype(int)

mri_feature_cols = [c for c in mri_raw.columns if c not in [id_col, mri_date_col]]
mri_feature_table = preprocess_mri_full(mri_raw[mri_feature_cols].copy())

aligned_mri_rows = []
for idx_match in match_idx:
    if pd.isna(idx_match):
        aligned_mri_rows.append(np.full(len(mri_feature_cols), MISSING_VALUE, dtype=np.float32))
    else:
        aligned_mri_rows.append(mri_feature_table.loc[int(idx_match)].values.astype(np.float32))

mrisbm_aligned = pd.DataFrame(aligned_mri_rows, columns=mri_feature_cols, index=uds_meta.index)

print("\nAligned MRI shape   :", mrisbm_aligned.shape)
print("MRI available count :", int(mri_available_mask.sum()))
print("MRI missing count   :", int((1 - mri_available_mask).sum()))


# ============================================================
# SAVE
# ============================================================

subject_id_out = uds_keep[[id_col]].copy().reset_index(drop=True)
visit_meta_out = uds_keep[[id_col, "_UDS_DATE_"]].copy().reset_index(drop=True)
label_out = uds_keep[["LABEL"]].copy().reset_index(drop=True)
mask_out = pd.DataFrame({"MRI_AVAILABLE": mri_available_mask.values.astype(int)})

uds_history = uds_history.reset_index(drop=True)
uds_survey = uds_survey.reset_index(drop=True)
uds_testing = uds_testing.reset_index(drop=True)
mrisbm_aligned = mrisbm_aligned.reset_index(drop=True)

subject_id_out.to_csv(os.path.join(OUT_DIR, "subject_id.csv"), index=False)
visit_meta_out.to_csv(os.path.join(OUT_DIR, "visit_meta.csv"), index=False)
label_out.to_csv(os.path.join(OUT_DIR, "label.csv"), index=False)
mask_out.to_csv(os.path.join(OUT_DIR, "mri_available_mask.csv"), index=False)

uds_history.to_csv(os.path.join(OUT_DIR, "uds_history.csv"), index=False)
uds_survey.to_csv(os.path.join(OUT_DIR, "uds_survey.csv"), index=False)
uds_testing.to_csv(os.path.join(OUT_DIR, "uds_testing.csv"), index=False)
mrisbm_aligned.to_csv(os.path.join(OUT_DIR, "mrisbm.csv"), index=False)

print("\nSaved files to:", OUT_DIR)
print("Files:")
print("  subject_id.csv")
print("  visit_meta.csv")
print("  label.csv")
print("  mri_available_mask.csv")
print("  uds_history.csv")
print("  uds_survey.csv")
print("  uds_testing.csv")
print("  mrisbm.csv")

diagnosis - all samples - knn imputation

In [3]:
# ============================================================
# COMPLETE REWRITE — v5 FAST:
#   5-FOLD STRATIFIED OUTER CV
#   + INNER STRATIFIED TRAIN/VAL SPLIT
#
# DATASET:
#   preprocessed_v3_for_joint_missingMRI_keepRawDims
#   Files:
#     subject_id.csv  — subject identifiers (row-aligned)
#     label.csv       — integer labels 0/1/2
#     uds_history.csv — already imputed + scaled
#     uds_survey.csv  — already imputed + scaled
#     uds_testing.csv — already imputed + scaled
#     mrisbm.csv      — already scaled; -999.0 sentinel = MRI missing
#
# MODALITIES:
#   UDS = concat(uds_history, uds_survey, uds_testing)  shape (N, D_uds)
#   MRI = mrisbm flat array                              shape (N, D_mri)
#   ALL rows kept — including MRI-missing rows.
#
# MISSING MRI HANDLING — KNN IMPUTATION WITHOUT DATA LEAKAGE:
#   MRI-missing rows have ALL 198 MRI columns set to -999.0.
#   Per-fold imputation procedure (matches reference multimodal codebase):
#     1. Replace -999.0 sentinel → np.nan
#     2. Pre-fill any remaining nan with per-column TRAIN means
#        (prevents KNNImputer from failing on all-nan columns)
#     3. Fit KNNImputer(n_neighbors=5, weights="distance") on
#        TRAIN rows only
#     4. Transform val and test with the fitted imputer
#     5. Final nan_to_num safety net
#   Distance weighting means structurally similar patients contribute
#   more, giving better reconstructions than median or uniform KNN.
#   Leakage guarantee: imputer is fit on train only; val/test only
#   ever see .transform().  Col means for pre-fill also from train only.
#
# SPEED IMPROVEMENTS vs v4:
#   1. Fold array caching  — precompute_fold_arrays() runs once before
#                            all experiments; 5 KNN fits total instead
#                            of one per runner per fold (85×)
#   2. UDS-only skip       — UDS experiments need no imputation at all;
#                            pure numpy slice with zero extra cost
#   3. Graph batch size    — 32 → 512 (16× fewer batches per epoch)
#   4. Epochs / patience   — max_epochs 50→30, patience 10→7 everywhere
#                            GTN pretraining 50→30 epochs
#                            MetaGraphClassifier patience 20→10
#
# TASK:
#   MULTICLASS DIAGNOSIS
#     class 0: LABEL 0  (original NACCUDSD in [1, 2])
#     class 1: LABEL 1  (original NACCUDSD == 3)
#     class 2: LABEL 2  (original NACCUDSD == 4)
#
# SPLIT DESIGN (PER OUTER FOLD):
#   test  = 20 % of full dataset
#   remaining 80 %:
#     val   = 12.5 % of remaining  = 10 % of total
#     train = 87.5 % of remaining  = 70 % of total
#
# EVALUATION:
#   Aggregate ALL out-of-fold predictions → compute metrics from
#   concatenated OOF array (not per-fold average).
#
# OUTPUTS:
#   multiclass_5fold_oof_predictions.csv
#   multiclass_5fold_final_summary.csv
# ============================================================

import os
import time
import copy
import random
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader

from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import (
    GCNConv,
    GATConv,
    TransformerConv,
    global_mean_pool,
    global_max_pool,
)

warnings.filterwarnings("ignore")

# ============================================================
# 0) OPTIONAL XGBOOST
# ============================================================

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False
    print("WARNING: xgboost not installed — XGBoost runs will be skipped.")

# ============================================================
# 1) GLOBAL CONSTANTS / DEVICE
# ============================================================

NUM_CLASSES    = 3
OUTER_N_SPLITS = 5
MISSING_VALUE  = -999.0

DATA_DIR     = "./preprocessed_v3_for_joint_missingMRI_keepRawDims"
MRI_ADJ_PATH = "./combined_adjacency_matrix.csv"

# Training hyper-parameters
GRAPH_BATCH_SIZE = 512
TAB_BATCH_SIZE   = 64
MAX_EPOCHS       = 30
PATIENCE         = 7
GTN_EPOCHS       = 30
META_PATIENCE    = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2) CLASS WEIGHT HELPERS
# ============================================================

def compute_class_weight_vector(y: np.ndarray,
                                num_classes: int = NUM_CLASSES) -> np.ndarray:
    y      = np.asarray(y).astype(int)
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    total  = counts.sum()
    w = np.zeros(num_classes, dtype=np.float32)
    for c in range(num_classes):
        if counts[c] > 0:
            w[c] = float(total / (num_classes * counts[c]))
    return w


def compute_class_weight_dict(y: np.ndarray,
                               num_classes: int = NUM_CLASSES
                               ) -> Dict[int, float]:
    v = compute_class_weight_vector(y, num_classes)
    return {c: float(v[c]) for c in range(num_classes) if v[c] > 0}


def compute_sample_weights(y: np.ndarray,
                           num_classes: int = NUM_CLASSES) -> np.ndarray:
    v = compute_class_weight_vector(y, num_classes)
    return v[np.asarray(y).astype(int)].astype(np.float32)


def class_weight_tensor(y: np.ndarray,
                        device=DEVICE,
                        num_classes: int = NUM_CLASSES) -> torch.Tensor:
    v = compute_class_weight_vector(y, num_classes)
    return torch.tensor(v, dtype=torch.float32, device=device)

# ============================================================
# 3) DATA LOADING
#    Keeps ALL rows including MRI-missing.
#    MRI-missing rows retain -999.0 — imputation happens
#    inside the pre-computed fold cache (Section 8).
# ============================================================

def load_v3_data(data_dir: str = DATA_DIR) -> dict:
    """
    Returns:
      ids          — subject IDs,          shape (N,)
      y            — integer labels 0/1/2, shape (N,)
      uds_arr      — float32 UDS,          shape (N, D_uds)
      mri_arr_raw  — float32 MRI,          shape (N, D_mri)
                     -999.0 where MRI was absent
      mri_missing  — bool mask,            shape (N,)
      uds_widths   — dict {history, survey, testing} column counts
      n_mri_nodes  — D_mri
    """
    subject_df = pd.read_csv(os.path.join(data_dir, "subject_id.csv"))
    label_df   = pd.read_csv(os.path.join(data_dir, "label.csv"))
    history_df = pd.read_csv(os.path.join(data_dir, "uds_history.csv"))
    survey_df  = pd.read_csv(os.path.join(data_dir, "uds_survey.csv"))
    testing_df = pd.read_csv(os.path.join(data_dir, "uds_testing.csv"))
    mri_df     = pd.read_csv(os.path.join(data_dir, "mrisbm.csv"))

    n = len(label_df)
    assert (len(subject_df) == len(history_df) == len(survey_df) ==
            len(testing_df) == len(mri_df) == n), \
        "Row-count mismatch across preprocessed files."

    id_col = subject_df.columns[0]
    ids    = subject_df[id_col].values
    y      = label_df["LABEL"].astype(int).values

    history_arr = history_df.values.astype(np.float32)
    survey_arr  = survey_df.values.astype(np.float32)
    testing_arr = testing_df.values.astype(np.float32)
    uds_arr     = np.concatenate(
        [history_arr, survey_arr, testing_arr], axis=1)

    uds_widths = {
        "history": history_arr.shape[1],
        "survey":  survey_arr.shape[1],
        "testing": testing_arr.shape[1],
    }

    mri_arr_raw = mri_df.values.astype(np.float32)
    mri_missing = np.all(mri_arr_raw == MISSING_VALUE, axis=1)

    n_miss = int(mri_missing.sum())
    print(f"[DATA] total rows           : {n}")
    print(f"[DATA] MRI-missing rows     : {n_miss}  "
          f"({100*n_miss/n:.1f} %)")
    print(f"[DATA] fully-aligned rows   : {n - n_miss}")
    print(f"[DATA] UDS dim              : {uds_arr.shape[1]}  "
          f"(history={uds_widths['history']}, "
          f"survey={uds_widths['survey']}, "
          f"testing={uds_widths['testing']})")
    print(f"[DATA] MRI dim              : {mri_arr_raw.shape[1]}")
    print(f"[DATA] Class distribution   : "
          f"{ {c: int((y==c).sum()) for c in range(NUM_CLASSES)} }")

    return dict(ids=ids, y=y, uds_arr=uds_arr,
                mri_arr_raw=mri_arr_raw, mri_missing=mri_missing,
                uds_widths=uds_widths,
                n_mri_nodes=int(mri_arr_raw.shape[1]))

# ============================================================
# 4) ADJACENCY BUILDERS
# ============================================================

def build_uds_structured_adj(uds_widths: dict) -> np.ndarray:
    """Intra-block fully-connected; history / survey / testing blocks."""
    n_hist = uds_widths["history"]
    n_surv = uds_widths["survey"]
    n_test = uds_widths["testing"]
    total  = n_hist + n_surv + n_test
    adj    = np.zeros((total, total), dtype=np.float32)
    for off, sz in zip([0, n_hist, n_hist + n_surv],
                       [n_hist, n_surv, n_test]):
        adj[off:off+sz, off:off+sz] = 1.0
    np.fill_diagonal(adj, 1.0)
    return adj


def build_mri_structured_adj(n_nodes: int,
                              adj_path: str = MRI_ADJ_PATH) -> np.ndarray:
    if os.path.exists(adj_path):
        try:
            adj = pd.read_csv(adj_path, index_col=0,
                              header=0).values.astype(float)
            if (adj.shape == (n_nodes, n_nodes) and
                    np.allclose(adj, adj.T)):
                print(f"[ADJ] Loaded MRI adjacency  shape={adj.shape}")
                return adj.astype(np.float32)
            print("[ADJ] MRI adj wrong shape/asymmetric — using identity.")
        except Exception as exc:
            print(f"[ADJ] Could not load adj: {exc} — using identity.")
    print(f"[ADJ] Identity adjacency for MRI  (n={n_nodes})")
    return np.eye(n_nodes, dtype=np.float32)


def build_identity_adj(n: int) -> np.ndarray:
    return np.eye(n, dtype=np.float32)


def build_bad_unstructured_adj(structured_adj: np.ndarray,
                                seed: int) -> np.ndarray:
    rng    = np.random.default_rng(seed)
    n      = structured_adj.shape[0]
    out    = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(out, 1.0)
    upper  = np.triu(structured_adj, k=1)
    n_edge = int((upper > 0).sum())
    ai, aj = np.triu_indices(n, k=1)
    ch     = rng.choice(len(ai), size=n_edge, replace=False)
    out[ai[ch], aj[ch]] = 1.0
    out[aj[ch], ai[ch]] = 1.0
    return out


def adjacency_to_edge_index(adj: np.ndarray) -> torch.Tensor:
    ei = np.array(np.nonzero(adj), dtype=np.int64)
    return torch.tensor(ei, dtype=torch.long)


def block_diag_adjacency(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    na, nb = a.shape[0], b.shape[0]
    out = np.zeros((na + nb, na + nb), dtype=float)
    out[:na, :na] = a
    out[na:, na:] = b
    return out

# ============================================================
# 5) METRICS
# ============================================================

def compute_multiclass_specificity(y_true, y_pred, nc=3):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(nc)))
    s  = []
    for c in range(nc):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - (cm[c, :].sum() - tp) - fp
        s.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return float(np.nanmean(s))


def compute_multiclass_sensitivity(y_true, y_pred, nc=3):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(nc)))
    s  = []
    for c in range(nc):
        tp = cm[c, c]; fn = cm[c, :].sum() - tp
        s.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
    return float(np.nanmean(s))


def compute_multiclass_auc(y_true, y_prob):
    try:
        yb = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        return float(roc_auc_score(yb, y_prob,
                                   multi_class="ovr", average="macro"))
    except Exception:
        return np.nan


def compute_binary_auc(y_true, y_prob):
    try:
        yb  = (np.asarray(y_true) != 0).astype(int)
        pp  = np.asarray(y_prob)[:, 1] + np.asarray(y_prob)[:, 2]
        return float(roc_auc_score(yb, pp)) \
            if len(np.unique(yb)) == 2 else np.nan
    except Exception:
        return np.nan


def sensitivity_at_80_spec(y_true, y_prob):
    try:
        yb  = (np.asarray(y_true) != 0).astype(int)
        sc  = np.asarray(y_prob)[:, 1] + np.asarray(y_prob)[:, 2]
        if len(np.unique(yb)) < 2:
            return dict(threshold=np.nan,
                        sensitivity=np.nan, specificity=np.nan)
        fpr, tpr, thr = roc_curve(yb, sc)
        spec = 1.0 - fpr
        idx  = int(np.argmin(np.abs(spec - 0.80)))
        return dict(threshold=float(thr[idx]),
                    sensitivity=float(tpr[idx]),
                    specificity=float(spec[idx]))
    except Exception:
        return dict(threshold=np.nan,
                    sensitivity=np.nan, specificity=np.nan)


def compute_final_metrics(y_true, y_pred, y_prob):
    s80 = sensitivity_at_80_spec(y_true, y_prob)
    return {
        "accuracy":                              accuracy_score(y_true, y_pred),
        "auc_macro_ovr":                         compute_multiclass_auc(y_true, y_prob),
        "auc_binary_collapse_debug":             compute_binary_auc(y_true, y_prob),
        "sensitivity_macro":                     compute_multiclass_sensitivity(y_true, y_pred, NUM_CLASSES),
        "specificity_macro":                     compute_multiclass_specificity(y_true, y_pred, NUM_CLASSES),
        "sensitivity_at_spec80_binary":          s80["sensitivity"],
        "achieved_specificity_at_spec80_binary": s80["specificity"],
        "threshold_for_spec80_binary":           s80["threshold"],
    }

# ============================================================
# 6) CV SPLITS
# ============================================================

def generate_cv_splits(y: np.ndarray,
                       n_splits: int = 5,
                       base_seed: int = 123) -> List[dict]:
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True,
                           random_state=base_seed)
    full = np.arange(len(y))
    out  = []
    for fold_id, (outer_tr, test_idx) in enumerate(skf.split(full, y)):
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=0.125,
            random_state=base_seed + fold_id)
        inner_tr, val_idx = next(sss.split(outer_tr, y[outer_tr]))
        out.append(dict(fold=fold_id,
                        train_idx=outer_tr[inner_tr],
                        val_idx=outer_tr[val_idx],
                        test_idx=test_idx))
    return out

# ============================================================
# 7) KNN IMPUTATION (MRI only, no leakage)
#
#   Matches the approach from the reference multimodal codebase:
#     1. Replace -999.0 sentinel → np.nan
#     2. Pre-fill any remaining nan with per-column TRAIN means
#        (handles columns that are entirely nan in a split, which
#        would otherwise cause KNNImputer to fail)
#     3. Fit KNNImputer(n_neighbors=5, weights="distance") on
#        TRAIN rows only — distance weighting means closer
#        neighbours contribute proportionally more, giving better
#        reconstructions than uniform averaging or median
#     4. Transform val and test with the fitted imputer
#     5. Final nan_to_num safety net for any residual nans
#
#   Leakage guarantee:
#     - KNNImputer.fit() sees only mri_train rows
#     - Column means used for pre-fill are also from train only
#     - Val and test are only ever passed to .transform()
#
#   Runtime note:
#     KNNImputer is O(N_train × N_query × D) at transform time.
#     With ~128K train rows × 198 features this will take several
#     minutes per fold — which is acceptable given fold caching
#     means it runs exactly 5 times total for the entire pipeline.
# ============================================================

KNN_N_NEIGHBORS = 5
KNN_WEIGHTS     = "distance"


def _sentinel_to_nan(a: np.ndarray) -> np.ndarray:
    """Replace MISSING_VALUE sentinel with np.nan. Returns float32 copy."""
    out = a.copy().astype(np.float32)
    out[out == MISSING_VALUE] = np.nan
    return out


def _prefill_with_train_means(tr: np.ndarray,
                               va: np.ndarray,
                               te: np.ndarray,
                               ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Fill any nan entries using per-column means computed from tr only.
    This is a safety step before KNN — KNNImputer cannot handle columns
    that are entirely nan, so we replace those with 0 after taking means.
    """
    col_means = np.nanmean(tr, axis=0)
    col_means = np.where(np.isnan(col_means), 0.0, col_means)

    def _fill(a):
        out  = a.copy()
        mask = np.isnan(out)
        if mask.any():
            out[mask] = np.take(col_means, np.where(mask)[1])
        return out

    return _fill(tr), _fill(va), _fill(te)


def impute_mri_fold(mri_train: np.ndarray,
                    mri_val:   np.ndarray,
                    mri_test:  np.ndarray,
                    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Full KNN imputation pipeline for one CV fold.
    Returns float32 arrays with no remaining nan / sentinel values.
    """
    # Step 1 — sentinel → nan
    tr_nan = _sentinel_to_nan(mri_train)
    va_nan = _sentinel_to_nan(mri_val)
    te_nan = _sentinel_to_nan(mri_test)

    # Step 2 — pre-fill with train column means (KNN safety net)
    tr_nan, va_nan, te_nan = _prefill_with_train_means(
        tr_nan, va_nan, te_nan)

    # Step 3 — fit KNN on TRAIN only
    imputer = KNNImputer(n_neighbors=KNN_N_NEIGHBORS,
                         weights=KNN_WEIGHTS)
    imputer.fit(tr_nan)

    # Step 4 — transform all three splits
    tr_imp = imputer.transform(tr_nan)
    va_imp = imputer.transform(va_nan)
    te_imp = imputer.transform(te_nan)

    # Step 5 — final safety net for any residual nans
    tr_imp = np.nan_to_num(tr_imp, nan=0.0).astype(np.float32)
    va_imp = np.nan_to_num(va_imp, nan=0.0).astype(np.float32)
    te_imp = np.nan_to_num(te_imp, nan=0.0).astype(np.float32)

    return tr_imp, va_imp, te_imp

# ============================================================
# 8) FOLD ARRAY CACHE
#
#   Runs ONCE before all experiments.
#   Produces 5 fold dicts, each containing:
#     X_uds_{tr,va,te}  — UDS slices (no imputation needed)
#     X_mri_{tr,va,te}  — MRI slices, KNN-imputed (train stats only)
#     y_{tr,va,te}       — label arrays
#     ids_te             — subject IDs for OOF rows
#
#   Every CV runner receives the pre-computed list and indexes by fold_id.
#   Total imputation cost: 5 fits instead of one per runner × per fold.
# ============================================================

def precompute_fold_arrays(X_uds:      np.ndarray,
                           X_mri_raw:  np.ndarray,
                           y:          np.ndarray,
                           ids:        np.ndarray,
                           cv_splits:  List[dict]) -> List[dict]:
    """
    Pre-impute MRI for every fold.  UDS is sliced without processing.
    """
    cache = []
    for split in cv_splits:
        fold_id = split["fold"]
        ti, vi, tei = (split["train_idx"],
                       split["val_idx"],
                       split["test_idx"])

        # UDS — pure slice
        X_uds_tr = X_uds[ti].astype(np.float32)
        X_uds_va = X_uds[vi].astype(np.float32)
        X_uds_te = X_uds[tei].astype(np.float32)

        # MRI — median impute, fit on train only
        X_mri_tr, X_mri_va, X_mri_te = impute_mri_fold(
            X_mri_raw[ti], X_mri_raw[vi], X_mri_raw[tei])

        cache.append(dict(
            fold=fold_id,
            X_uds_tr=X_uds_tr, X_uds_va=X_uds_va, X_uds_te=X_uds_te,
            X_mri_tr=X_mri_tr, X_mri_va=X_mri_va, X_mri_te=X_mri_te,
            y_tr=y[ti], y_va=y[vi], y_te=y[tei],
            ids_te=ids[tei],
        ))
        print(f"[CACHE] fold {fold_id} imputed  "
              f"train={len(ti)}  val={len(vi)}  test={len(tei)}")

    return cache

# ============================================================
# 9) TORCH DATASETS
# ============================================================

class ArrayDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================================
# 10) NON-GRAPH MODELS
# ============================================================

class MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128),
                 dropout=0.2, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN1DNet(nn.Module):
    def __init__(self, seq_len, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.classifier(
            self.features(x.unsqueeze(1)).squeeze(-1))


class TabularTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128,
                 dropout=0.1, num_classes=3):
        super().__init__()
        self.proj    = nn.Linear(1, d_model)
        enc          = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="relu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.cls     = nn.Linear(d_model, num_classes)

    def forward(self, x):
        return self.cls(
            self.encoder(self.proj(x.unsqueeze(-1))).mean(dim=1))

# ============================================================
# 11) GRAPH MODELS
# ============================================================

class GCNGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels=64,
                 dropout=0.2, num_classes=3):
        super().__init__()
        self.conv1   = GCNConv(in_channels, hidden_channels)
        self.conv2   = GCNConv(hidden_channels, hidden_channels)
        self.lin     = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        return self.lin(global_mean_pool(x, b))


class GATGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels=32,
                 heads=4, dropout=0.2, num_classes=3):
        super().__init__()
        self.gat1    = GATConv(in_channels, hidden_channels,
                               heads=heads, dropout=dropout)
        self.gat2    = GATConv(hidden_channels * heads, hidden_channels,
                               heads=1, dropout=dropout)
        self.lin     = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.gat1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.gat2(x, ei))
        return self.lin(global_mean_pool(x, b))

# ============================================================
# 12) TRAINING / INFERENCE HELPERS
# ============================================================

def _train_torch(model, train_loader, val_loader, cw,
                 lr=1e-3, wd=1e-4,
                 max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model     = model.to(DEVICE)
    cw        = cw.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_loss, best_state, pat = float("inf"), None, 0

    for _ in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl.append(criterion(model(xb.to(DEVICE)),
                                    yb.to(DEVICE)).item())
        v = float(np.mean(vl))
        if v < best_loss:
            best_loss  = v
            best_state = {k: w.cpu().clone()
                          for k, w in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def _predict_torch(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            lg = model(xb.to(DEVICE))
            probs.extend(torch.softmax(lg, 1).cpu().numpy())
            preds.extend(torch.argmax(lg, 1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(trues), np.array(preds), np.array(probs)


def _train_pyg(model, train_loader, val_loader, cw,
               lr=1e-3, wd=1e-4,
               max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model     = model.to(DEVICE)
    cw        = cw.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_loss, best_state, pat = float("inf"), None, 0

    for _ in range(max_epochs):
        model.train()
        for b in train_loader:
            b = b.to(DEVICE)
            opt.zero_grad()
            criterion(model(b), b.y.view(-1)).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(DEVICE)
                vl.append(criterion(model(b),
                                    b.y.view(-1)).item())
        v = float(np.mean(vl))
        if v < best_loss:
            best_loss  = v
            best_state = {k: w.cpu().clone()
                          for k, w in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def _predict_pyg(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for b in loader:
            b  = b.to(DEVICE)
            lg = model(b)
            probs.extend(torch.softmax(lg, 1).cpu().numpy())
            preds.extend(torch.argmax(lg, 1).cpu().numpy())
            trues.extend(b.y.view(-1).cpu().numpy())
    return np.array(trues), np.array(preds), np.array(probs)

# ============================================================
# 13) GRAPH CONSTRUCTION
#     Each feature column = 1 node with 1 feature.
#     edge_index is built once per adjacency and reused.
#     All values are already imputed — no sentinel values here.
# ============================================================

def _make_flat_graphs(X: np.ndarray, y: np.ndarray,
                      edge_index: torch.Tensor) -> List[Data]:
    return [
        Data(x=torch.tensor(X[i].reshape(-1, 1), dtype=torch.float32),
             edge_index=edge_index,
             y=torch.tensor([int(y[i])], dtype=torch.long))
        for i in range(len(X))
    ]


def build_graphs_from_fold(X_tr, X_va, X_te,
                            y_tr, y_va, y_te,
                            adjacency: np.ndarray
                            ) -> Tuple[List, List, List]:
    ei = adjacency_to_edge_index(adjacency)
    return (_make_flat_graphs(X_tr, y_tr, ei),
            _make_flat_graphs(X_va, y_va, ei),
            _make_flat_graphs(X_te, y_te, ei))


def build_early_fusion_graphs(X_uds_tr, X_uds_va, X_uds_te,
                               X_mri_tr, X_mri_va, X_mri_te,
                               y_tr, y_va, y_te,
                               uds_adj: np.ndarray,
                               mri_adj: np.ndarray
                               ) -> Tuple[List, List, List]:
    ei = adjacency_to_edge_index(block_diag_adjacency(uds_adj, mri_adj))

    def _make(Xu, Xm, y):
        return [
            Data(x=torch.tensor(
                     np.concatenate([Xu[i], Xm[i]]).reshape(-1, 1),
                     dtype=torch.float32),
                 edge_index=ei,
                 y=torch.tensor([int(y[i])], dtype=torch.long))
            for i in range(len(y))
        ]

    return _make(X_uds_tr, X_mri_tr, y_tr), \
           _make(X_uds_va, X_mri_va, y_va), \
           _make(X_uds_te, X_mri_te, y_te)

# ============================================================
# 14) NON-GRAPH MODEL RUNNERS
# ============================================================

def _run_lr(X_tr, X_te, y_tr, y_te):
    m = LogisticRegression(
        max_iter=3000, solver="lbfgs", multi_class="multinomial",
        class_weight=compute_class_weight_dict(y_tr))
    m.fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)
    return y_te, np.argmax(prob, 1), prob


def _run_xgb(X_tr, X_te, y_tr, y_te):
    if not XGB_AVAILABLE:
        prob = np.full((len(X_te), NUM_CLASSES), np.nan)
        return y_te, np.zeros(len(X_te), dtype=int), prob
    m = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="multi:softprob", num_class=NUM_CLASSES,
        eval_metric="mlogloss", random_state=0)
    m.fit(X_tr, y_tr, sample_weight=compute_sample_weights(y_tr))
    prob = m.predict_proba(X_te)
    return y_te, np.argmax(prob, 1), prob


def _run_deep(model_name, X_tr, X_va, X_te, y_tr, y_va, y_te):
    trl = TorchDataLoader(ArrayDataset(X_tr, y_tr),
                          batch_size=TAB_BATCH_SIZE, shuffle=True)
    val = TorchDataLoader(ArrayDataset(X_va, y_va),
                          batch_size=TAB_BATCH_SIZE, shuffle=False)
    tel = TorchDataLoader(ArrayDataset(X_te, y_te),
                          batch_size=TAB_BATCH_SIZE, shuffle=False)
    d = X_tr.shape[1]
    if model_name == "MLP":
        m = MLPNet(d, num_classes=NUM_CLASSES)
    elif model_name == "CNN":
        m = CNN1DNet(d, num_classes=NUM_CLASSES)
    elif model_name == "Transformer":
        m = TabularTransformer(d, d_model=64, nhead=4,
                               num_layers=2, num_classes=NUM_CLASSES)
    else:
        raise ValueError(model_name)
    m = _train_torch(m, trl, val, class_weight_tensor(y_tr))
    return _predict_torch(m, tel)


def run_non_graph_model(model_name,
                        X_tr, X_va, X_te,
                        y_tr, y_va, y_te):
    if model_name == "LogisticRegression":
        return _run_lr(X_tr, X_te, y_tr, y_te)
    if model_name == "XGBoost":
        return _run_xgb(X_tr, X_te, y_tr, y_te)
    if model_name in ("MLP", "CNN", "Transformer"):
        return _run_deep(model_name, X_tr, X_va, X_te,
                         y_tr, y_va, y_te)
    raise ValueError(f"Unknown model: {model_name}")


def run_graph_model(model_name, tr_g, va_g, te_g,
                    in_channels: int, y_tr: np.ndarray):
    trl = PyGDataLoader(tr_g, batch_size=GRAPH_BATCH_SIZE, shuffle=True)
    val = PyGDataLoader(va_g, batch_size=GRAPH_BATCH_SIZE, shuffle=False)
    tel = PyGDataLoader(te_g, batch_size=GRAPH_BATCH_SIZE, shuffle=False)

    if model_name == "GCN":
        m = GCNGraphClassifier(in_channels, 64, num_classes=NUM_CLASSES)
    elif model_name == "GAT":
        m = GATGraphClassifier(in_channels, 32, heads=4,
                               num_classes=NUM_CLASSES)
    else:
        raise ValueError(model_name)

    m = _train_pyg(m, trl, val, class_weight_tensor(y_tr))
    return _predict_pyg(m, tel)

# ============================================================
# 15) PROPOSED MODEL — GTN pretraining + MetaGraphClassifier
# ============================================================

class ImprovedGTN(nn.Module):
    def __init__(self, num_node_features, hidden_channels,
                 num_classes, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(num_node_features,
                                     hidden_channels, heads=heads,
                                     dropout=0.1)
        self.conv2 = TransformerConv(hidden_channels * heads,
                                     hidden_channels, heads=heads,
                                     dropout=0.1)
        self.conv3 = TransformerConv(hidden_channels * heads,
                                     32, heads=1, dropout=0.1)
        self.bn    = nn.LayerNorm(32)
        self.fc    = nn.Linear(32, num_classes)
        self.drop  = nn.Dropout(p=0.3)

    def forward(self, data, return_edge_weights=False):
        x, ei = data.x, data.edge_index
        batch  = (data.batch if hasattr(data, "batch")
                  else torch.zeros(x.size(0), dtype=torch.long,
                                   device=x.device))
        x = F.relu(self.conv1(x, ei)); x = self.drop(x)
        x = F.relu(self.conv2(x, ei)); x = self.drop(x)

        if return_edge_weights:
            x, (ei_out, ew_out) = self.conv3(
                x, ei, return_attention_weights=True)
            ei_out = ei_out.to(x.device)
            ew_out = ew_out.to(x.device)
        else:
            x = self.conv3(x, ei)

        x      = self.bn(F.relu(x))
        pooled = global_max_pool(x, batch)
        if return_edge_weights:
            return pooled, x, ei_out, ew_out
        return self.fc(pooled)


class MetaGraphClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = TransformerConv(input_dim, hidden_dim,
                                     heads=2, dropout=0.1)
        self.conv2 = TransformerConv(hidden_dim * 2, hidden_dim,
                                     heads=1, dropout=0.1)
        self.cls   = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.relu(self.conv2(x, ei))
        return self.cls(global_mean_pool(x, b))


def _xavier_init(m):
    for layer in m.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


def _train_gnns(loaders, gnns, y_tr):
    cw  = class_weight_tensor(y_tr)
    crit = nn.CrossEntropyLoss(weight=cw)
    for g in gnns:
        g.to(DEVICE)
    opts = [torch.optim.Adam(g.parameters(), lr=0.002,
                             weight_decay=1e-4) for g in gnns]
    for _ in range(GTN_EPOCHS):
        for loader, gnn, opt in zip(loaders, gnns, opts):
            gnn.train()
            for b in loader:
                b = b.to(DEVICE)
                opt.zero_grad()
                crit(gnn(b), b.y.view(-1)).backward()
                opt.step()


def _extract_features(dataset, gnn):
    nf, ei, ew = [], [], []
    gnn.eval()
    with torch.no_grad():
        for d in dataset:
            d = d.to(DEVICE)
            _, n, e_i, e_w = gnn(d, return_edge_weights=True)
            nf.append(n.cpu())
            ei.append(e_i.cpu())
            ew.append(e_w.cpu())
    return nf, ei, ew


def _build_new_graphs(nf, ei, ew, y):
    out = []
    for x, e_i, e_w, label in zip(nf, ei, ew, y):
        if e_w.ndim > 1:
            e_w = e_w.mean(dim=1)
        out.append(Data(x=x, edge_index=e_i, edge_attr=e_w,
                        y=torch.tensor([int(label)],
                                       dtype=torch.long)))
    return out


def _merge_graphs(g1, g2):
    off = g1.x.size(0)
    cx  = torch.cat([g1.x, g2.x], dim=0)
    cei = torch.cat([g1.edge_index, g2.edge_index + off], dim=1)
    cea = (torch.cat([g1.edge_attr, g2.edge_attr], dim=0)
           if (getattr(g1, "edge_attr", None) is not None and
               getattr(g2, "edge_attr", None) is not None)
           else None)
    return Data(x=cx, edge_index=cei, edge_attr=cea, y=g1.y)


def _norm_graph_features(tr, va, te):
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(
        torch.cat([d.x for d in tr], 0).numpy())

    def _tx(ds):
        out = []
        for d in ds:
            dc   = copy.deepcopy(d)
            dc.x = torch.tensor(
                sc.transform(dc.x.numpy()), dtype=torch.float32)
            out.append(dc)
        return out

    return _tx(tr), _tx(va), _tx(te)


def run_proposed_fusion(tr_uds, va_uds, te_uds,
                        tr_mri, va_mri, te_mri,
                        y_tr, y_va, y_te):
    tr_uds, va_uds, te_uds = _norm_graph_features(tr_uds, va_uds, te_uds)
    tr_mri, va_mri, te_mri = _norm_graph_features(tr_mri, va_mri, te_mri)

    trl_u = PyGDataLoader(tr_uds, batch_size=GRAPH_BATCH_SIZE,
                          shuffle=False)
    trl_m = PyGDataLoader(tr_mri, batch_size=GRAPH_BATCH_SIZE,
                          shuffle=False)

    gnn_u = ImprovedGTN(1, 64, NUM_CLASSES)
    gnn_m = ImprovedGTN(1, 64, NUM_CLASSES)
    _xavier_init(gnn_u); _xavier_init(gnn_m)
    _train_gnns([trl_u, trl_m], [gnn_u, gnn_m], y_tr)

    def _feats(ds, gnn):
        return _extract_features(ds, gnn)

    tr_nu, tr_eu, tr_wu = _feats(tr_uds, gnn_u)
    va_nu, va_eu, va_wu = _feats(va_uds, gnn_u)
    te_nu, te_eu, te_wu = _feats(te_uds, gnn_u)
    tr_nm, tr_em, tr_wm = _feats(tr_mri, gnn_m)
    va_nm, va_em, va_wm = _feats(va_mri, gnn_m)
    te_nm, te_em, te_wm = _feats(te_mri, gnn_m)

    m_tr = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(tr_nu, tr_eu, tr_wu, y_tr),
        _build_new_graphs(tr_nm, tr_em, tr_wm, y_tr))]
    m_va = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(va_nu, va_eu, va_wu, y_va),
        _build_new_graphs(va_nm, va_em, va_wm, y_va))]
    m_te = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(te_nu, te_eu, te_wu, y_te),
        _build_new_graphs(te_nm, te_em, te_wm, y_te))]

    tr_b  = Batch.from_data_list(m_tr).to(DEVICE)
    va_b  = Batch.from_data_list(m_va).to(DEVICE)
    te_b  = Batch.from_data_list(m_te).to(DEVICE)
    tr_lb = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    va_lb = torch.tensor(y_va, dtype=torch.long, device=DEVICE)

    cw   = class_weight_tensor(y_tr)
    crit = nn.CrossEntropyLoss(weight=cw)
    meta = MetaGraphClassifier(32, 64, NUM_CLASSES).to(DEVICE)
    opt  = torch.optim.Adam(meta.parameters(), lr=0.01)
    bvl, bst, pat = float("inf"), None, 0

    for _ in range(MAX_EPOCHS):
        meta.train()
        opt.zero_grad()
        crit(meta(tr_b), tr_lb).backward()
        opt.step()
        meta.eval()
        with torch.no_grad():
            vl = crit(meta(va_b), va_lb).item()
        if vl < bvl:
            bvl = vl
            bst = copy.deepcopy(meta.state_dict())
            pat = 0
        else:
            pat += 1
            if pat >= META_PATIENCE:
                break

    if bst:
        meta.load_state_dict(bst)
    meta.eval()
    with torch.no_grad():
        out   = meta(te_b)
        probs = torch.softmax(out, 1).cpu().numpy()
        preds = torch.argmax(out, 1).cpu().numpy()
    return np.array(y_te), preds, probs

# ============================================================
# 16) OOF BOOKKEEPING
# ============================================================

def build_oof_rows(ids, y_true, y_pred, y_prob,
                   fold_id, cohort, exp_family,
                   exp_name, setting, model_name):
    return [{
        "cohort":            cohort,
        "experiment_family": exp_family,
        "experiment_name":   exp_name,
        "setting":           setting,
        "model":             model_name,
        "fold":              fold_id,
        "subject_id":        ids[i],
        "y_true":            int(y_true[i]),
        "y_pred":            int(y_pred[i]),
        "prob_class_0":      float(y_prob[i, 0]),
        "prob_class_1":      float(y_prob[i, 1]),
        "prob_class_2":      float(y_prob[i, 2]),
    } for i in range(len(y_true))]


def summarize_oof(oof_df, cohort, exp_family,
                  exp_name, setting, model_name):
    yt = oof_df["y_true"].values.astype(int)
    yp = oof_df["y_pred"].values.astype(int)
    yb = oof_df[["prob_class_0",
                  "prob_class_1",
                  "prob_class_2"]].values.astype(float)
    m  = compute_final_metrics(yt, yp, yb)
    m.update(dict(cohort=cohort, experiment_family=exp_family,
                  experiment_name=exp_name, setting=setting,
                  model=model_name,
                  n_total_oof_samples=len(oof_df)))
    return m

# ============================================================
# 17) CV EXPERIMENT RUNNERS
#
#   Each runner receives `fold_cache` (list of 5 pre-computed dicts)
#   instead of raw arrays.  No imputation happens inside runners.
#   UDS-only runners skip MRI arrays entirely.
# ============================================================

def _log(yt, yp, yb, tag):
    fm = compute_final_metrics(yt, yp, yb)
    print(f"{tag} "
          f"acc={fm['accuracy']:.4f} "
          f"auc_macro={fm['auc_macro_ovr']:.4f} "
          f"auc_bin={fm['auc_binary_collapse_debug']:.4f}")


# ---------- UNIMODAL NON-GRAPH ----------

def run_cv_unimodal_non_graph(model_name, modality_name,
                               fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(100 + fid)
        t0  = time.time()

        if modality_name == "UDS":
            Xtr, Xva, Xte = fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"]
        else:
            Xtr, Xva, Xte = fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"]

        yt, yp, yb = run_non_graph_model(
            model_name, Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "UNIMODAL", "UNIMODAL_NON_GRAPH",
            modality_name, model_name))
        _log(yt, yp, yb,
             f"[UNI][NON_GRAPH][{modality_name}][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "UNIMODAL",
                            "UNIMODAL_NON_GRAPH", modality_name,
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- UNIMODAL GRAPH ----------

def run_cv_unimodal_graph(model_name, modality_name,
                           fold_cache, cv_splits,
                           adjacency, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(200 + fid)
        t0  = time.time()

        if modality_name == "UDS":
            Xtr, Xva, Xte = fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"]
        else:
            Xtr, Xva, Xte = fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"]

        tr_g, va_g, te_g = build_graphs_from_fold(
            Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"], adjacency)
        yt, yp, yb = run_graph_model(
            model_name, tr_g, va_g, te_g,
            in_channels=1, y_tr=fd["y_tr"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "UNIMODAL", "UNIMODAL_GRAPH",
            f"{modality_name}_{structure_name}", model_name))
        _log(yt, yp, yb,
             f"[UNI][GRAPH][{modality_name}][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "UNIMODAL",
                            "UNIMODAL_GRAPH",
                            f"{modality_name}_{structure_name}",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- EARLY FUSION NON-GRAPH ----------

def run_cv_early_fusion_non_graph(model_name, fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(300 + fid)
        t0  = time.time()

        Xtr = np.concatenate([fd["X_uds_tr"], fd["X_mri_tr"]], axis=1)
        Xva = np.concatenate([fd["X_uds_va"], fd["X_mri_va"]], axis=1)
        Xte = np.concatenate([fd["X_uds_te"], fd["X_mri_te"]], axis=1)

        yt, yp, yb = run_non_graph_model(
            model_name, Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "FUSION", "EARLY_FUSION_NON_GRAPH",
            "early", model_name))
        _log(yt, yp, yb,
             f"[EARLY_FUSION][NON_GRAPH][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "EARLY_FUSION_NON_GRAPH", "early",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- LATE FUSION NON-GRAPH ----------

def run_cv_late_fusion_non_graph(model_name, fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(400 + fid)
        t0  = time.time()

        _, _, p_uds = run_non_graph_model(
            model_name,
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"])
        _, _, p_mri = run_non_graph_model(
            model_name,
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"])

        prob = 0.5 * (p_uds + p_mri)
        pred = np.argmax(prob, axis=1)

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], fd["y_te"], pred, prob, fid,
            "ALIGNED", "FUSION", "LATE_FUSION_NON_GRAPH",
            "late", model_name))
        _log(fd["y_te"], pred, prob,
             f"[LATE_FUSION][NON_GRAPH][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "LATE_FUSION_NON_GRAPH", "late",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- EARLY FUSION GRAPH ----------

def run_cv_early_fusion_graph(model_name, fold_cache, cv_splits,
                               uds_adj, mri_adj, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(500 + fid)
        t0  = time.time()

        tr_g, va_g, te_g = build_early_fusion_graphs(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"],
            uds_adj, mri_adj)
        yt, yp, yb = run_graph_model(
            model_name, tr_g, va_g, te_g,
            in_channels=1, y_tr=fd["y_tr"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "FUSION", "EARLY_FUSION_GRAPH",
            f"early_{structure_name}", model_name))
        _log(yt, yp, yb,
             f"[EARLY_FUSION][GRAPH][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "EARLY_FUSION_GRAPH",
                            f"early_{structure_name}", model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- LATE FUSION GRAPH ----------

def run_cv_late_fusion_graph(model_name, fold_cache, cv_splits,
                              uds_adj, mri_adj, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(600 + fid)
        t0  = time.time()

        tr_ug, va_ug, te_ug = build_graphs_from_fold(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], uds_adj)
        tr_mg, va_mg, te_mg = build_graphs_from_fold(
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], mri_adj)

        _, _, p_uds = run_graph_model(
            model_name, tr_ug, va_ug, te_ug,
            in_channels=1, y_tr=fd["y_tr"])
        _, _, p_mri = run_graph_model(
            model_name, tr_mg, va_mg, te_mg,
            in_channels=1, y_tr=fd["y_tr"])

        prob = 0.5 * (p_uds + p_mri)
        pred = np.argmax(prob, axis=1)

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], fd["y_te"], pred, prob, fid,
            "ALIGNED", "FUSION", "LATE_FUSION_GRAPH",
            f"late_{structure_name}", model_name))
        _log(fd["y_te"], pred, prob,
             f"[LATE_FUSION][GRAPH][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "LATE_FUSION_GRAPH",
                            f"late_{structure_name}", model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- PROPOSED FUSION ----------

def _get_proposed_adjs(setting, uds_s, mri_s, fid):
    if setting == "structured":
        return uds_s, mri_s
    if setting == "previous_unstructured":
        return build_identity_adj(uds_s.shape[0]), \
               build_identity_adj(mri_s.shape[0])
    if setting == "new_unstructured_bad":
        return build_bad_unstructured_adj(uds_s, 1000 + fid), \
               build_bad_unstructured_adj(mri_s, 2000 + fid)
    raise ValueError(setting)


def run_cv_proposed_fusion(setting_name,
                            fold_cache, cv_splits,
                            uds_struct_adj, mri_struct_adj):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(700 + fid)
        t0  = time.time()

        uds_adj, mri_adj = _get_proposed_adjs(
            setting_name, uds_struct_adj, mri_struct_adj, fid)

        tr_ug, va_ug, te_ug = build_graphs_from_fold(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], uds_adj)
        tr_mg, va_mg, te_mg = build_graphs_from_fold(
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], mri_adj)

        yt, yp, yb = run_proposed_fusion(
            tr_ug, va_ug, te_ug,
            tr_mg, va_mg, te_mg,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        mname = f"ImprovedGTN_MetaGraphClassifier_{setting_name}"
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "PROPOSED", "PROPOSED_FUSION",
            setting_name, mname))
        _log(yt, yp, yb,
             f"[PROPOSED][{setting_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    mname   = f"ImprovedGTN_MetaGraphClassifier_{setting_name}"
    summary = summarize_oof(oof_df, "ALIGNED", "PROPOSED",
                            "PROPOSED_FUSION", setting_name, mname)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary

# ============================================================
# 18) MAIN
# ============================================================

def main():
    all_oof, all_summary = [], []

    # ----------------------------------------------------------
    # LOAD — all 183K+ rows, -999 sentinel intact in MRI
    # ----------------------------------------------------------
    data        = load_v3_data(DATA_DIR)
    y           = data["y"]
    ids         = data["ids"]
    X_uds       = data["uds_arr"]
    X_mri_raw   = data["mri_arr_raw"]
    uds_widths  = data["uds_widths"]
    n_mri_nodes = data["n_mri_nodes"]

    print(f"\n[MAIN] N={len(y)}  "
          f"UDS={X_uds.shape[1]}  MRI={X_mri_raw.shape[1]}\n")

    # ----------------------------------------------------------
    # ADJACENCY
    # ----------------------------------------------------------
    uds_adj_s = build_uds_structured_adj(uds_widths)
    uds_adj_u = build_identity_adj(X_uds.shape[1])
    mri_adj_s = build_mri_structured_adj(n_mri_nodes, MRI_ADJ_PATH)
    mri_adj_u = build_identity_adj(n_mri_nodes)

    # ----------------------------------------------------------
    # CV SPLITS  (full dataset, stratified)
    # ----------------------------------------------------------
    cv_splits = generate_cv_splits(y, n_splits=OUTER_N_SPLITS,
                                   base_seed=123)

    # ----------------------------------------------------------
    # PRE-COMPUTE FOLD ARRAYS  (5 median fits, done once)
    # ----------------------------------------------------------
    print("\n[CACHE] Pre-computing fold arrays (median imputation)...")
    fold_cache = precompute_fold_arrays(
        X_uds, X_mri_raw, y, ids, cv_splits)
    print("[CACHE] Done.\n")

    # ----------------------------------------------------------
    # NON-GRAPH UNIMODAL
    # ----------------------------------------------------------
    non_graph_models = ["LogisticRegression", "MLP",
                        "XGBoost", "CNN", "Transformer"]

    for mn in non_graph_models:
        if mn == "XGBoost" and not XGB_AVAILABLE:
            continue
        for mod in ("UDS", "MRI"):
            oof, s = run_cv_unimodal_non_graph(
                mn, mod, fold_cache, cv_splits)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # GRAPH UNIMODAL
    # ----------------------------------------------------------
    graph_models = ["GCN", "GAT"]

    for gm in graph_models:
        for sname, adj in [("structured",   uds_adj_s),
                            ("unstructured", uds_adj_u)]:
            oof, s = run_cv_unimodal_graph(
                gm, "UDS", fold_cache, cv_splits, adj, sname)
            all_oof.append(oof); all_summary.append(s)

        for sname, adj in [("structured",   mri_adj_s),
                            ("unstructured", mri_adj_u)]:
            oof, s = run_cv_unimodal_graph(
                gm, "MRI", fold_cache, cv_splits, adj, sname)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # NON-GRAPH FUSION
    # ----------------------------------------------------------
    for mn in non_graph_models:
        if mn == "XGBoost" and not XGB_AVAILABLE:
            continue
        oof, s = run_cv_early_fusion_non_graph(
            mn, fold_cache, cv_splits)
        all_oof.append(oof); all_summary.append(s)

        oof, s = run_cv_late_fusion_non_graph(
            mn, fold_cache, cv_splits)
        all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # GRAPH FUSION
    # ----------------------------------------------------------
    for gm in graph_models:
        for sname, ua, ma in [
                ("structured",   uds_adj_s, mri_adj_s),
                ("unstructured", uds_adj_u, mri_adj_u)]:
            oof, s = run_cv_early_fusion_graph(
                gm, fold_cache, cv_splits, ua, ma, sname)
            all_oof.append(oof); all_summary.append(s)

            oof, s = run_cv_late_fusion_graph(
                gm, fold_cache, cv_splits, ua, ma, sname)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # PROPOSED FUSION
    # ----------------------------------------------------------
    for setting in ["structured",
                    "previous_unstructured",
                    "new_unstructured_bad"]:
        oof, s = run_cv_proposed_fusion(
            setting, fold_cache, cv_splits,
            uds_adj_s, mri_adj_s)
        all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # SAVE
    # ----------------------------------------------------------
    oof_all    = pd.concat(all_oof, axis=0, ignore_index=True)
    summary_df = pd.DataFrame(all_summary)

    col_order = [
        "cohort", "experiment_family", "experiment_name", "setting",
        "model", "n_total_oof_samples",
        "accuracy", "auc_macro_ovr", "auc_binary_collapse_debug",
        "sensitivity_macro", "specificity_macro",
        "sensitivity_at_spec80_binary",
        "achieved_specificity_at_spec80_binary",
        "threshold_for_spec80_binary",
        "avg_fold_time_seconds",
    ]
    summary_df = summary_df[col_order]

    print("\n" + "=" * 120)
    print("FINAL OOF SUMMARY")
    print("=" * 120)
    print(summary_df.to_string(index=False))

    oof_all.to_csv("multiclass_5fold_oof_predictions.csv", index=False)
    summary_df.to_csv("multiclass_5fold_final_summary.csv", index=False)
    print("\nSaved: multiclass_5fold_oof_predictions.csv")
    print("       multiclass_5fold_final_summary.csv")


if __name__ == "__main__":
    main()

Using device: cpu
[DATA] total rows           : 185831
[DATA] MRI-missing rows     : 183452  (98.7 %)
[DATA] fully-aligned rows   : 2379
[DATA] UDS dim              : 134  (history=66, survey=39, testing=29)
[DATA] MRI dim              : 198
[DATA] Class distribution   : {0: 97491, 1: 32490, 2: 55850}

[MAIN] N=185831  UDS=134  MRI=198

[ADJ] Identity adjacency for MRI  (n=198)

[CACHE] Pre-computing fold arrays (median imputation)...
[CACHE] fold 0 imputed  train=130081  val=18583  test=37167
[CACHE] fold 1 imputed  train=130081  val=18584  test=37166
[CACHE] fold 2 imputed  train=130081  val=18584  test=37166
[CACHE] fold 3 imputed  train=130081  val=18584  test=37166
[CACHE] fold 4 imputed  train=130081  val=18584  test=37166
[CACHE] Done.

[UNI][NON_GRAPH][UDS][LogisticRegression][f=0] acc=0.7667 auc_macro=0.8942 auc_bin=0.9205
[UNI][NON_GRAPH][UDS][LogisticRegression][f=1] acc=0.7706 auc_macro=0.8965 auc_bin=0.9237
[UNI][NON_GRAPH][UDS][LogisticRegression][f=2] acc=0.7744 auc_macr

KeyboardInterrupt: 

diagnosis - all samples - mice imputation

In [5]:
# ============================================================
# COMPLETE REWRITE — v5 FAST:
#   5-FOLD STRATIFIED OUTER CV
#   + INNER STRATIFIED TRAIN/VAL SPLIT
#
# DATASET:
#   preprocessed_v3_for_joint_missingMRI_keepRawDims
#   Files:
#     subject_id.csv  — subject identifiers (row-aligned)
#     label.csv       — integer labels 0/1/2
#     uds_history.csv — already imputed + scaled
#     uds_survey.csv  — already imputed + scaled
#     uds_testing.csv — already imputed + scaled
#     mrisbm.csv      — already scaled; -999.0 sentinel = MRI missing
#
# MODALITIES:
#   UDS = concat(uds_history, uds_survey, uds_testing)  shape (N, D_uds)
#   MRI = mrisbm flat array                              shape (N, D_mri)
#   ALL rows kept — including MRI-missing rows.
#
# MISSING MRI HANDLING — KNN IMPUTATION WITHOUT DATA LEAKAGE:
#   MRI-missing rows have ALL 198 MRI columns set to -999.0.
#   Per-fold imputation procedure (matches reference multimodal codebase):
#     1. Replace -999.0 sentinel → np.nan
#     2. Pre-fill any remaining nan with per-column TRAIN means
#        (prevents KNNImputer from failing on all-nan columns)
#     3. Fit KNNImputer(n_neighbors=5, weights="distance") on
#        TRAIN rows only
#     4. Transform val and test with the fitted imputer
#     5. Final nan_to_num safety net
#   Distance weighting means structurally similar patients contribute
#   more, giving better reconstructions than median or uniform KNN.
#   Leakage guarantee: imputer is fit on train only; val/test only
#   ever see .transform().  Col means for pre-fill also from train only.
#
# SPEED IMPROVEMENTS vs v4:
#   1. Fold array caching  — precompute_fold_arrays() runs once before
#                            all experiments; 5 MICE fits total instead
#                            of one per runner per fold (85×)
#   2. UDS-only skip       — UDS experiments need no imputation at all;
#                            pure numpy slice with zero extra cost
#   3. Graph batch size    — 32 → 512 (16× fewer batches per epoch)
#   4. Epochs / patience   — max_epochs 50→30, patience 10→7 everywhere
#                            GTN pretraining 50→30 epochs
#                            MetaGraphClassifier patience 20→10
#   5. MICE speed tuning   — n_nearest_features=30 limits each column
#                            regression to 30 predictors instead of 198
#
# TASK:
#   MULTICLASS DIAGNOSIS
#     class 0: LABEL 0  (original NACCUDSD in [1, 2])
#     class 1: LABEL 1  (original NACCUDSD == 3)
#     class 2: LABEL 2  (original NACCUDSD == 4)
#
# SPLIT DESIGN (PER OUTER FOLD):
#   test  = 20 % of full dataset
#   remaining 80 %:
#     val   = 12.5 % of remaining  = 10 % of total
#     train = 87.5 % of remaining  = 70 % of total
#
# EVALUATION:
#   Aggregate ALL out-of-fold predictions → compute metrics from
#   concatenated OOF array (not per-fold average).
#
# OUTPUTS:
#   multiclass_5fold_oof_predictions.csv
#   multiclass_5fold_final_summary.csv
# ============================================================

import os
import time
import copy
import random
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader

from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import (
    GCNConv,
    GATConv,
    TransformerConv,
    global_mean_pool,
    global_max_pool,
)

warnings.filterwarnings("ignore")

# ============================================================
# 0) OPTIONAL XGBOOST
# ============================================================

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False
    print("WARNING: xgboost not installed — XGBoost runs will be skipped.")

# ============================================================
# 1) GLOBAL CONSTANTS / DEVICE
# ============================================================

NUM_CLASSES    = 3
OUTER_N_SPLITS = 5
MISSING_VALUE  = -999.0

DATA_DIR     = "./preprocessed_v3_for_joint_missingMRI_keepRawDims"
MRI_ADJ_PATH = "./combined_adjacency_matrix.csv"

# Training hyper-parameters
GRAPH_BATCH_SIZE = 512
TAB_BATCH_SIZE   = 64
MAX_EPOCHS       = 30
PATIENCE         = 7
GTN_EPOCHS       = 30
META_PATIENCE    = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2) CLASS WEIGHT HELPERS
# ============================================================

def compute_class_weight_vector(y: np.ndarray,
                                num_classes: int = NUM_CLASSES) -> np.ndarray:
    y      = np.asarray(y).astype(int)
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    total  = counts.sum()
    w = np.zeros(num_classes, dtype=np.float32)
    for c in range(num_classes):
        if counts[c] > 0:
            w[c] = float(total / (num_classes * counts[c]))
    return w


def compute_class_weight_dict(y: np.ndarray,
                               num_classes: int = NUM_CLASSES
                               ) -> Dict[int, float]:
    v = compute_class_weight_vector(y, num_classes)
    return {c: float(v[c]) for c in range(num_classes) if v[c] > 0}


def compute_sample_weights(y: np.ndarray,
                           num_classes: int = NUM_CLASSES) -> np.ndarray:
    v = compute_class_weight_vector(y, num_classes)
    return v[np.asarray(y).astype(int)].astype(np.float32)


def class_weight_tensor(y: np.ndarray,
                        device=DEVICE,
                        num_classes: int = NUM_CLASSES) -> torch.Tensor:
    v = compute_class_weight_vector(y, num_classes)
    return torch.tensor(v, dtype=torch.float32, device=device)

# ============================================================
# 3) DATA LOADING
#    Keeps ALL rows including MRI-missing.
#    MRI-missing rows retain -999.0 — imputation happens
#    inside the pre-computed fold cache (Section 8).
# ============================================================

def load_v3_data(data_dir: str = DATA_DIR) -> dict:
    """
    Returns:
      ids          — subject IDs,          shape (N,)
      y            — integer labels 0/1/2, shape (N,)
      uds_arr      — float32 UDS,          shape (N, D_uds)
      mri_arr_raw  — float32 MRI,          shape (N, D_mri)
                     -999.0 where MRI was absent
      mri_missing  — bool mask,            shape (N,)
      uds_widths   — dict {history, survey, testing} column counts
      n_mri_nodes  — D_mri
    """
    subject_df = pd.read_csv(os.path.join(data_dir, "subject_id.csv"))
    label_df   = pd.read_csv(os.path.join(data_dir, "label.csv"))
    history_df = pd.read_csv(os.path.join(data_dir, "uds_history.csv"))
    survey_df  = pd.read_csv(os.path.join(data_dir, "uds_survey.csv"))
    testing_df = pd.read_csv(os.path.join(data_dir, "uds_testing.csv"))
    mri_df     = pd.read_csv(os.path.join(data_dir, "mrisbm.csv"))

    n = len(label_df)
    assert (len(subject_df) == len(history_df) == len(survey_df) ==
            len(testing_df) == len(mri_df) == n), \
        "Row-count mismatch across preprocessed files."

    id_col = subject_df.columns[0]
    ids    = subject_df[id_col].values
    y      = label_df["LABEL"].astype(int).values

    history_arr = history_df.values.astype(np.float32)
    survey_arr  = survey_df.values.astype(np.float32)
    testing_arr = testing_df.values.astype(np.float32)
    uds_arr     = np.concatenate(
        [history_arr, survey_arr, testing_arr], axis=1)

    uds_widths = {
        "history": history_arr.shape[1],
        "survey":  survey_arr.shape[1],
        "testing": testing_arr.shape[1],
    }

    mri_arr_raw = mri_df.values.astype(np.float32)
    mri_missing = np.all(mri_arr_raw == MISSING_VALUE, axis=1)

    n_miss = int(mri_missing.sum())
    print(f"[DATA] total rows           : {n}")
    print(f"[DATA] MRI-missing rows     : {n_miss}  "
          f"({100*n_miss/n:.1f} %)")
    print(f"[DATA] fully-aligned rows   : {n - n_miss}")
    print(f"[DATA] UDS dim              : {uds_arr.shape[1]}  "
          f"(history={uds_widths['history']}, "
          f"survey={uds_widths['survey']}, "
          f"testing={uds_widths['testing']})")
    print(f"[DATA] MRI dim              : {mri_arr_raw.shape[1]}")
    print(f"[DATA] Class distribution   : "
          f"{ {c: int((y==c).sum()) for c in range(NUM_CLASSES)} }")

    return dict(ids=ids, y=y, uds_arr=uds_arr,
                mri_arr_raw=mri_arr_raw, mri_missing=mri_missing,
                uds_widths=uds_widths,
                n_mri_nodes=int(mri_arr_raw.shape[1]))

# ============================================================
# 4) ADJACENCY BUILDERS
# ============================================================

def build_uds_structured_adj(uds_widths: dict) -> np.ndarray:
    """Intra-block fully-connected; history / survey / testing blocks."""
    n_hist = uds_widths["history"]
    n_surv = uds_widths["survey"]
    n_test = uds_widths["testing"]
    total  = n_hist + n_surv + n_test
    adj    = np.zeros((total, total), dtype=np.float32)
    for off, sz in zip([0, n_hist, n_hist + n_surv],
                       [n_hist, n_surv, n_test]):
        adj[off:off+sz, off:off+sz] = 1.0
    np.fill_diagonal(adj, 1.0)
    return adj


def build_mri_structured_adj(n_nodes: int,
                              adj_path: str = MRI_ADJ_PATH) -> np.ndarray:
    if os.path.exists(adj_path):
        try:
            adj = pd.read_csv(adj_path, index_col=0,
                              header=0).values.astype(float)
            if (adj.shape == (n_nodes, n_nodes) and
                    np.allclose(adj, adj.T)):
                print(f"[ADJ] Loaded MRI adjacency  shape={adj.shape}")
                return adj.astype(np.float32)
            print("[ADJ] MRI adj wrong shape/asymmetric — using identity.")
        except Exception as exc:
            print(f"[ADJ] Could not load adj: {exc} — using identity.")
    print(f"[ADJ] Identity adjacency for MRI  (n={n_nodes})")
    return np.eye(n_nodes, dtype=np.float32)


def build_identity_adj(n: int) -> np.ndarray:
    return np.eye(n, dtype=np.float32)


def build_bad_unstructured_adj(structured_adj: np.ndarray,
                                seed: int) -> np.ndarray:
    rng    = np.random.default_rng(seed)
    n      = structured_adj.shape[0]
    out    = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(out, 1.0)
    upper  = np.triu(structured_adj, k=1)
    n_edge = int((upper > 0).sum())
    ai, aj = np.triu_indices(n, k=1)
    ch     = rng.choice(len(ai), size=n_edge, replace=False)
    out[ai[ch], aj[ch]] = 1.0
    out[aj[ch], ai[ch]] = 1.0
    return out


def adjacency_to_edge_index(adj: np.ndarray) -> torch.Tensor:
    ei = np.array(np.nonzero(adj), dtype=np.int64)
    return torch.tensor(ei, dtype=torch.long)


def block_diag_adjacency(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    na, nb = a.shape[0], b.shape[0]
    out = np.zeros((na + nb, na + nb), dtype=float)
    out[:na, :na] = a
    out[na:, na:] = b
    return out

# ============================================================
# 5) METRICS
# ============================================================

def compute_multiclass_specificity(y_true, y_pred, nc=3):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(nc)))
    s  = []
    for c in range(nc):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - (cm[c, :].sum() - tp) - fp
        s.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return float(np.nanmean(s))


def compute_multiclass_sensitivity(y_true, y_pred, nc=3):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(nc)))
    s  = []
    for c in range(nc):
        tp = cm[c, c]; fn = cm[c, :].sum() - tp
        s.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
    return float(np.nanmean(s))


def compute_multiclass_auc(y_true, y_prob):
    try:
        yb = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        return float(roc_auc_score(yb, y_prob,
                                   multi_class="ovr", average="macro"))
    except Exception:
        return np.nan


def compute_binary_auc(y_true, y_prob):
    try:
        yb  = (np.asarray(y_true) != 0).astype(int)
        pp  = np.asarray(y_prob)[:, 1] + np.asarray(y_prob)[:, 2]
        return float(roc_auc_score(yb, pp)) \
            if len(np.unique(yb)) == 2 else np.nan
    except Exception:
        return np.nan


def sensitivity_at_80_spec(y_true, y_prob):
    try:
        yb  = (np.asarray(y_true) != 0).astype(int)
        sc  = np.asarray(y_prob)[:, 1] + np.asarray(y_prob)[:, 2]
        if len(np.unique(yb)) < 2:
            return dict(threshold=np.nan,
                        sensitivity=np.nan, specificity=np.nan)
        fpr, tpr, thr = roc_curve(yb, sc)
        spec = 1.0 - fpr
        idx  = int(np.argmin(np.abs(spec - 0.80)))
        return dict(threshold=float(thr[idx]),
                    sensitivity=float(tpr[idx]),
                    specificity=float(spec[idx]))
    except Exception:
        return dict(threshold=np.nan,
                    sensitivity=np.nan, specificity=np.nan)


def compute_final_metrics(y_true, y_pred, y_prob):
    s80 = sensitivity_at_80_spec(y_true, y_prob)
    return {
        "accuracy":                              accuracy_score(y_true, y_pred),
        "auc_macro_ovr":                         compute_multiclass_auc(y_true, y_prob),
        "auc_binary_collapse_debug":             compute_binary_auc(y_true, y_prob),
        "sensitivity_macro":                     compute_multiclass_sensitivity(y_true, y_pred, NUM_CLASSES),
        "specificity_macro":                     compute_multiclass_specificity(y_true, y_pred, NUM_CLASSES),
        "sensitivity_at_spec80_binary":          s80["sensitivity"],
        "achieved_specificity_at_spec80_binary": s80["specificity"],
        "threshold_for_spec80_binary":           s80["threshold"],
    }

# ============================================================
# 6) CV SPLITS
# ============================================================

def generate_cv_splits(y: np.ndarray,
                       n_splits: int = 5,
                       base_seed: int = 123) -> List[dict]:
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True,
                           random_state=base_seed)
    full = np.arange(len(y))
    out  = []
    for fold_id, (outer_tr, test_idx) in enumerate(skf.split(full, y)):
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=0.125,
            random_state=base_seed + fold_id)
        inner_tr, val_idx = next(sss.split(outer_tr, y[outer_tr]))
        out.append(dict(fold=fold_id,
                        train_idx=outer_tr[inner_tr],
                        val_idx=outer_tr[val_idx],
                        test_idx=test_idx))
    return out

# ============================================================
# 7) MICE IMPUTATION (MRI only, no leakage)
#
#   Method: Multiple Imputation by Chained Equations (MICE)
#   via sklearn IterativeImputer with BayesianRidge estimator.
#
#   Why MICE is stronger than KNN here:
#     KNN on fully-missing rows computes distances in mean-imputed
#     space, so neighbours are nearly indistinguishable — the
#     "distance" weighting is almost meaningless.  MICE instead
#     iteratively regresses each MRI column on all other MRI
#     columns, learning the full inter-column covariance structure
#     from the 1.3% of complete rows.  Because brain morphometry
#     features are highly correlated (adjacent regions co-vary),
#     these regressions are informative and produce much better
#     imputations than neighbour averaging.
#
#   Leakage guarantee — all three levels are train-only:
#     1. IterativeImputer.fit() sees ONLY mri_train rows
#     2. The BayesianRidge regressors for each column are learned
#        from train rows only
#     3. The initial fill values (column means) are from train only
#     4. Val and test rows are only ever passed to .transform()
#        — sklearn's transform uses the fitted regressors to
#        predict missing values; it never updates them
#
#   Speed settings:
#     max_iter=10          — 10 full cycles through all 198 cols
#     n_nearest_features=30 — each col's regressor uses only its
#                             30 most correlated predictors,
#                             cutting cost ~7× vs using all 198
#     initial_strategy="mean" — warm-start from col means
#     random_state=0       — reproducible
# ============================================================

MICE_MAX_ITER         = 10
MICE_N_NEAREST_FEATS  = 30
MICE_RANDOM_STATE     = 0


def _sentinel_to_nan(a: np.ndarray) -> np.ndarray:
    """Replace MISSING_VALUE sentinel with np.nan. Returns float64 copy
    (IterativeImputer requires float64 internally)."""
    out = a.copy().astype(np.float64)
    out[out == MISSING_VALUE] = np.nan
    return out


def impute_mri_fold(mri_train: np.ndarray,
                    mri_val:   np.ndarray,
                    mri_test:  np.ndarray,
                    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    MICE imputation pipeline for one CV fold.
    Returns float32 arrays with no remaining nan / sentinel values.

    Leakage proof:
      - IterativeImputer.fit()  → mri_train only
      - IterativeImputer.transform() → applied to val and test
        using the already-fitted BayesianRidge regressors; no
        regressors are updated during transform
    """
    # Step 1 — sentinel → nan  (float64 for sklearn)
    tr_nan = _sentinel_to_nan(mri_train)
    va_nan = _sentinel_to_nan(mri_val)
    te_nan = _sentinel_to_nan(mri_test)

    # Step 2 — fit MICE on TRAIN only
    #   BayesianRidge is the default estimator — it handles
    #   colinear features robustly via automatic regularisation,
    #   which is ideal for correlated brain morphometry columns.
    imputer = IterativeImputer(
        estimator=None,            # default = BayesianRidge
        max_iter=MICE_MAX_ITER,
        n_nearest_features=MICE_N_NEAREST_FEATS,
        initial_strategy="mean",   # warm-start from train col means
        imputation_order="ascending",
        random_state=MICE_RANDOM_STATE,
        verbose=0,
    )
    imputer.fit(tr_nan)            # fit on TRAIN only

    # Step 3 — transform all three splits with fitted imputer
    tr_imp = imputer.transform(tr_nan)
    va_imp = imputer.transform(va_nan)
    te_imp = imputer.transform(te_nan)

    # Step 4 — safety net for any residual nans (edge case: column
    #   entirely nan in train → BayesianRidge has no signal → nan)
    tr_imp = np.nan_to_num(tr_imp, nan=0.0).astype(np.float32)
    va_imp = np.nan_to_num(va_imp, nan=0.0).astype(np.float32)
    te_imp = np.nan_to_num(te_imp, nan=0.0).astype(np.float32)

    return tr_imp, va_imp, te_imp

# ============================================================
# 8) FOLD ARRAY CACHE
#
#   Runs ONCE before all experiments.
#   Produces 5 fold dicts, each containing:
#     X_uds_{tr,va,te}  — UDS slices (no imputation needed)
#     X_mri_{tr,va,te}  — MRI slices, MICE-imputed (train stats only)
#     y_{tr,va,te}       — label arrays
#     ids_te             — subject IDs for OOF rows
#
#   Every CV runner receives the pre-computed list and indexes by fold_id.
#   Total imputation cost: 5 fits instead of one per runner × per fold.
# ============================================================

def precompute_fold_arrays(X_uds:      np.ndarray,
                           X_mri_raw:  np.ndarray,
                           y:          np.ndarray,
                           ids:        np.ndarray,
                           cv_splits:  List[dict]) -> List[dict]:
    """
    Pre-impute MRI for every fold.  UDS is sliced without processing.
    """
    cache = []
    for split in cv_splits:
        fold_id = split["fold"]
        ti, vi, tei = (split["train_idx"],
                       split["val_idx"],
                       split["test_idx"])

        # UDS — pure slice
        X_uds_tr = X_uds[ti].astype(np.float32)
        X_uds_va = X_uds[vi].astype(np.float32)
        X_uds_te = X_uds[tei].astype(np.float32)

        # MRI — MICE impute, fit on train only
        X_mri_tr, X_mri_va, X_mri_te = impute_mri_fold(
            X_mri_raw[ti], X_mri_raw[vi], X_mri_raw[tei])

        cache.append(dict(
            fold=fold_id,
            X_uds_tr=X_uds_tr, X_uds_va=X_uds_va, X_uds_te=X_uds_te,
            X_mri_tr=X_mri_tr, X_mri_va=X_mri_va, X_mri_te=X_mri_te,
            y_tr=y[ti], y_va=y[vi], y_te=y[tei],
            ids_te=ids[tei],
        ))
        print(f"[CACHE] fold {fold_id} imputed  "
              f"train={len(ti)}  val={len(vi)}  test={len(tei)}")

    return cache

# ============================================================
# 9) TORCH DATASETS
# ============================================================

class ArrayDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================================
# 10) NON-GRAPH MODELS
# ============================================================

class MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128),
                 dropout=0.2, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN1DNet(nn.Module):
    def __init__(self, seq_len, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.classifier(
            self.features(x.unsqueeze(1)).squeeze(-1))


class TabularTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128,
                 dropout=0.1, num_classes=3):
        super().__init__()
        self.proj    = nn.Linear(1, d_model)
        enc          = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="relu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.cls     = nn.Linear(d_model, num_classes)

    def forward(self, x):
        return self.cls(
            self.encoder(self.proj(x.unsqueeze(-1))).mean(dim=1))

# ============================================================
# 11) GRAPH MODELS
# ============================================================

class GCNGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels=64,
                 dropout=0.2, num_classes=3):
        super().__init__()
        self.conv1   = GCNConv(in_channels, hidden_channels)
        self.conv2   = GCNConv(hidden_channels, hidden_channels)
        self.lin     = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        return self.lin(global_mean_pool(x, b))


class GATGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels=32,
                 heads=4, dropout=0.2, num_classes=3):
        super().__init__()
        self.gat1    = GATConv(in_channels, hidden_channels,
                               heads=heads, dropout=dropout)
        self.gat2    = GATConv(hidden_channels * heads, hidden_channels,
                               heads=1, dropout=dropout)
        self.lin     = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.gat1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.gat2(x, ei))
        return self.lin(global_mean_pool(x, b))

# ============================================================
# 12) TRAINING / INFERENCE HELPERS
# ============================================================

def _train_torch(model, train_loader, val_loader, cw,
                 lr=1e-3, wd=1e-4,
                 max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model     = model.to(DEVICE)
    cw        = cw.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_loss, best_state, pat = float("inf"), None, 0

    for _ in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl.append(criterion(model(xb.to(DEVICE)),
                                    yb.to(DEVICE)).item())
        v = float(np.mean(vl))
        if v < best_loss:
            best_loss  = v
            best_state = {k: w.cpu().clone()
                          for k, w in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def _predict_torch(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            lg = model(xb.to(DEVICE))
            probs.extend(torch.softmax(lg, 1).cpu().numpy())
            preds.extend(torch.argmax(lg, 1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(trues), np.array(preds), np.array(probs)


def _train_pyg(model, train_loader, val_loader, cw,
               lr=1e-3, wd=1e-4,
               max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model     = model.to(DEVICE)
    cw        = cw.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_loss, best_state, pat = float("inf"), None, 0

    for _ in range(max_epochs):
        model.train()
        for b in train_loader:
            b = b.to(DEVICE)
            opt.zero_grad()
            criterion(model(b), b.y.view(-1)).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(DEVICE)
                vl.append(criterion(model(b),
                                    b.y.view(-1)).item())
        v = float(np.mean(vl))
        if v < best_loss:
            best_loss  = v
            best_state = {k: w.cpu().clone()
                          for k, w in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def _predict_pyg(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for b in loader:
            b  = b.to(DEVICE)
            lg = model(b)
            probs.extend(torch.softmax(lg, 1).cpu().numpy())
            preds.extend(torch.argmax(lg, 1).cpu().numpy())
            trues.extend(b.y.view(-1).cpu().numpy())
    return np.array(trues), np.array(preds), np.array(probs)

# ============================================================
# 13) GRAPH CONSTRUCTION
#     Each feature column = 1 node with 1 feature.
#     edge_index is built once per adjacency and reused.
#     All values are already imputed — no sentinel values here.
# ============================================================

def _make_flat_graphs(X: np.ndarray, y: np.ndarray,
                      edge_index: torch.Tensor) -> List[Data]:
    return [
        Data(x=torch.tensor(X[i].reshape(-1, 1), dtype=torch.float32),
             edge_index=edge_index,
             y=torch.tensor([int(y[i])], dtype=torch.long))
        for i in range(len(X))
    ]


def build_graphs_from_fold(X_tr, X_va, X_te,
                            y_tr, y_va, y_te,
                            adjacency: np.ndarray
                            ) -> Tuple[List, List, List]:
    ei = adjacency_to_edge_index(adjacency)
    return (_make_flat_graphs(X_tr, y_tr, ei),
            _make_flat_graphs(X_va, y_va, ei),
            _make_flat_graphs(X_te, y_te, ei))


def build_early_fusion_graphs(X_uds_tr, X_uds_va, X_uds_te,
                               X_mri_tr, X_mri_va, X_mri_te,
                               y_tr, y_va, y_te,
                               uds_adj: np.ndarray,
                               mri_adj: np.ndarray
                               ) -> Tuple[List, List, List]:
    ei = adjacency_to_edge_index(block_diag_adjacency(uds_adj, mri_adj))

    def _make(Xu, Xm, y):
        return [
            Data(x=torch.tensor(
                     np.concatenate([Xu[i], Xm[i]]).reshape(-1, 1),
                     dtype=torch.float32),
                 edge_index=ei,
                 y=torch.tensor([int(y[i])], dtype=torch.long))
            for i in range(len(y))
        ]

    return _make(X_uds_tr, X_mri_tr, y_tr), \
           _make(X_uds_va, X_mri_va, y_va), \
           _make(X_uds_te, X_mri_te, y_te)

# ============================================================
# 14) NON-GRAPH MODEL RUNNERS
# ============================================================

def _run_lr(X_tr, X_te, y_tr, y_te):
    m = LogisticRegression(
        max_iter=3000, solver="lbfgs", multi_class="multinomial",
        class_weight=compute_class_weight_dict(y_tr))
    m.fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)
    return y_te, np.argmax(prob, 1), prob


def _run_xgb(X_tr, X_te, y_tr, y_te):
    if not XGB_AVAILABLE:
        prob = np.full((len(X_te), NUM_CLASSES), np.nan)
        return y_te, np.zeros(len(X_te), dtype=int), prob
    m = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="multi:softprob", num_class=NUM_CLASSES,
        eval_metric="mlogloss", random_state=0)
    m.fit(X_tr, y_tr, sample_weight=compute_sample_weights(y_tr))
    prob = m.predict_proba(X_te)
    return y_te, np.argmax(prob, 1), prob


def _run_deep(model_name, X_tr, X_va, X_te, y_tr, y_va, y_te):
    trl = TorchDataLoader(ArrayDataset(X_tr, y_tr),
                          batch_size=TAB_BATCH_SIZE, shuffle=True)
    val = TorchDataLoader(ArrayDataset(X_va, y_va),
                          batch_size=TAB_BATCH_SIZE, shuffle=False)
    tel = TorchDataLoader(ArrayDataset(X_te, y_te),
                          batch_size=TAB_BATCH_SIZE, shuffle=False)
    d = X_tr.shape[1]
    if model_name == "MLP":
        m = MLPNet(d, num_classes=NUM_CLASSES)
    elif model_name == "CNN":
        m = CNN1DNet(d, num_classes=NUM_CLASSES)
    elif model_name == "Transformer":
        m = TabularTransformer(d, d_model=64, nhead=4,
                               num_layers=2, num_classes=NUM_CLASSES)
    else:
        raise ValueError(model_name)
    m = _train_torch(m, trl, val, class_weight_tensor(y_tr))
    return _predict_torch(m, tel)


def run_non_graph_model(model_name,
                        X_tr, X_va, X_te,
                        y_tr, y_va, y_te):
    if model_name == "LogisticRegression":
        return _run_lr(X_tr, X_te, y_tr, y_te)
    if model_name == "XGBoost":
        return _run_xgb(X_tr, X_te, y_tr, y_te)
    if model_name in ("MLP", "CNN", "Transformer"):
        return _run_deep(model_name, X_tr, X_va, X_te,
                         y_tr, y_va, y_te)
    raise ValueError(f"Unknown model: {model_name}")


def run_graph_model(model_name, tr_g, va_g, te_g,
                    in_channels: int, y_tr: np.ndarray):
    trl = PyGDataLoader(tr_g, batch_size=GRAPH_BATCH_SIZE, shuffle=True)
    val = PyGDataLoader(va_g, batch_size=GRAPH_BATCH_SIZE, shuffle=False)
    tel = PyGDataLoader(te_g, batch_size=GRAPH_BATCH_SIZE, shuffle=False)

    if model_name == "GCN":
        m = GCNGraphClassifier(in_channels, 64, num_classes=NUM_CLASSES)
    elif model_name == "GAT":
        m = GATGraphClassifier(in_channels, 32, heads=4,
                               num_classes=NUM_CLASSES)
    else:
        raise ValueError(model_name)

    m = _train_pyg(m, trl, val, class_weight_tensor(y_tr))
    return _predict_pyg(m, tel)

# ============================================================
# 15) PROPOSED MODEL — GTN pretraining + MetaGraphClassifier
# ============================================================

class ImprovedGTN(nn.Module):
    def __init__(self, num_node_features, hidden_channels,
                 num_classes, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(num_node_features,
                                     hidden_channels, heads=heads,
                                     dropout=0.1)
        self.conv2 = TransformerConv(hidden_channels * heads,
                                     hidden_channels, heads=heads,
                                     dropout=0.1)
        self.conv3 = TransformerConv(hidden_channels * heads,
                                     32, heads=1, dropout=0.1)
        self.bn    = nn.LayerNorm(32)
        self.fc    = nn.Linear(32, num_classes)
        self.drop  = nn.Dropout(p=0.3)

    def forward(self, data, return_edge_weights=False):
        x, ei = data.x, data.edge_index
        batch  = (data.batch if hasattr(data, "batch")
                  else torch.zeros(x.size(0), dtype=torch.long,
                                   device=x.device))
        x = F.relu(self.conv1(x, ei)); x = self.drop(x)
        x = F.relu(self.conv2(x, ei)); x = self.drop(x)

        if return_edge_weights:
            x, (ei_out, ew_out) = self.conv3(
                x, ei, return_attention_weights=True)
            ei_out = ei_out.to(x.device)
            ew_out = ew_out.to(x.device)
        else:
            x = self.conv3(x, ei)

        x      = self.bn(F.relu(x))
        pooled = global_max_pool(x, batch)
        if return_edge_weights:
            return pooled, x, ei_out, ew_out
        return self.fc(pooled)


class MetaGraphClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = TransformerConv(input_dim, hidden_dim,
                                     heads=2, dropout=0.1)
        self.conv2 = TransformerConv(hidden_dim * 2, hidden_dim,
                                     heads=1, dropout=0.1)
        self.cls   = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.relu(self.conv2(x, ei))
        return self.cls(global_mean_pool(x, b))


def _xavier_init(m):
    for layer in m.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


def _train_gnns(loaders, gnns, y_tr):
    cw  = class_weight_tensor(y_tr)
    crit = nn.CrossEntropyLoss(weight=cw)
    for g in gnns:
        g.to(DEVICE)
    opts = [torch.optim.Adam(g.parameters(), lr=0.002,
                             weight_decay=1e-4) for g in gnns]
    for _ in range(GTN_EPOCHS):
        for loader, gnn, opt in zip(loaders, gnns, opts):
            gnn.train()
            for b in loader:
                b = b.to(DEVICE)
                opt.zero_grad()
                crit(gnn(b), b.y.view(-1)).backward()
                opt.step()


def _extract_features(dataset, gnn):
    nf, ei, ew = [], [], []
    gnn.eval()
    with torch.no_grad():
        for d in dataset:
            d = d.to(DEVICE)
            _, n, e_i, e_w = gnn(d, return_edge_weights=True)
            nf.append(n.cpu())
            ei.append(e_i.cpu())
            ew.append(e_w.cpu())
    return nf, ei, ew


def _build_new_graphs(nf, ei, ew, y):
    out = []
    for x, e_i, e_w, label in zip(nf, ei, ew, y):
        if e_w.ndim > 1:
            e_w = e_w.mean(dim=1)
        out.append(Data(x=x, edge_index=e_i, edge_attr=e_w,
                        y=torch.tensor([int(label)],
                                       dtype=torch.long)))
    return out


def _merge_graphs(g1, g2):
    off = g1.x.size(0)
    cx  = torch.cat([g1.x, g2.x], dim=0)
    cei = torch.cat([g1.edge_index, g2.edge_index + off], dim=1)
    cea = (torch.cat([g1.edge_attr, g2.edge_attr], dim=0)
           if (getattr(g1, "edge_attr", None) is not None and
               getattr(g2, "edge_attr", None) is not None)
           else None)
    return Data(x=cx, edge_index=cei, edge_attr=cea, y=g1.y)


def _norm_graph_features(tr, va, te):
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(
        torch.cat([d.x for d in tr], 0).numpy())

    def _tx(ds):
        out = []
        for d in ds:
            dc   = copy.deepcopy(d)
            dc.x = torch.tensor(
                sc.transform(dc.x.numpy()), dtype=torch.float32)
            out.append(dc)
        return out

    return _tx(tr), _tx(va), _tx(te)


def run_proposed_fusion(tr_uds, va_uds, te_uds,
                        tr_mri, va_mri, te_mri,
                        y_tr, y_va, y_te):
    tr_uds, va_uds, te_uds = _norm_graph_features(tr_uds, va_uds, te_uds)
    tr_mri, va_mri, te_mri = _norm_graph_features(tr_mri, va_mri, te_mri)

    trl_u = PyGDataLoader(tr_uds, batch_size=GRAPH_BATCH_SIZE,
                          shuffle=False)
    trl_m = PyGDataLoader(tr_mri, batch_size=GRAPH_BATCH_SIZE,
                          shuffle=False)

    gnn_u = ImprovedGTN(1, 64, NUM_CLASSES)
    gnn_m = ImprovedGTN(1, 64, NUM_CLASSES)
    _xavier_init(gnn_u); _xavier_init(gnn_m)
    _train_gnns([trl_u, trl_m], [gnn_u, gnn_m], y_tr)

    def _feats(ds, gnn):
        return _extract_features(ds, gnn)

    tr_nu, tr_eu, tr_wu = _feats(tr_uds, gnn_u)
    va_nu, va_eu, va_wu = _feats(va_uds, gnn_u)
    te_nu, te_eu, te_wu = _feats(te_uds, gnn_u)
    tr_nm, tr_em, tr_wm = _feats(tr_mri, gnn_m)
    va_nm, va_em, va_wm = _feats(va_mri, gnn_m)
    te_nm, te_em, te_wm = _feats(te_mri, gnn_m)

    m_tr = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(tr_nu, tr_eu, tr_wu, y_tr),
        _build_new_graphs(tr_nm, tr_em, tr_wm, y_tr))]
    m_va = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(va_nu, va_eu, va_wu, y_va),
        _build_new_graphs(va_nm, va_em, va_wm, y_va))]
    m_te = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(te_nu, te_eu, te_wu, y_te),
        _build_new_graphs(te_nm, te_em, te_wm, y_te))]

    tr_b  = Batch.from_data_list(m_tr).to(DEVICE)
    va_b  = Batch.from_data_list(m_va).to(DEVICE)
    te_b  = Batch.from_data_list(m_te).to(DEVICE)
    tr_lb = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    va_lb = torch.tensor(y_va, dtype=torch.long, device=DEVICE)

    cw   = class_weight_tensor(y_tr)
    crit = nn.CrossEntropyLoss(weight=cw)
    meta = MetaGraphClassifier(32, 64, NUM_CLASSES).to(DEVICE)
    opt  = torch.optim.Adam(meta.parameters(), lr=0.01)
    bvl, bst, pat = float("inf"), None, 0

    for _ in range(MAX_EPOCHS):
        meta.train()
        opt.zero_grad()
        crit(meta(tr_b), tr_lb).backward()
        opt.step()
        meta.eval()
        with torch.no_grad():
            vl = crit(meta(va_b), va_lb).item()
        if vl < bvl:
            bvl = vl
            bst = copy.deepcopy(meta.state_dict())
            pat = 0
        else:
            pat += 1
            if pat >= META_PATIENCE:
                break

    if bst:
        meta.load_state_dict(bst)
    meta.eval()
    with torch.no_grad():
        out   = meta(te_b)
        probs = torch.softmax(out, 1).cpu().numpy()
        preds = torch.argmax(out, 1).cpu().numpy()
    return np.array(y_te), preds, probs

# ============================================================
# 16) OOF BOOKKEEPING
# ============================================================

def build_oof_rows(ids, y_true, y_pred, y_prob,
                   fold_id, cohort, exp_family,
                   exp_name, setting, model_name):
    return [{
        "cohort":            cohort,
        "experiment_family": exp_family,
        "experiment_name":   exp_name,
        "setting":           setting,
        "model":             model_name,
        "fold":              fold_id,
        "subject_id":        ids[i],
        "y_true":            int(y_true[i]),
        "y_pred":            int(y_pred[i]),
        "prob_class_0":      float(y_prob[i, 0]),
        "prob_class_1":      float(y_prob[i, 1]),
        "prob_class_2":      float(y_prob[i, 2]),
    } for i in range(len(y_true))]


def summarize_oof(oof_df, cohort, exp_family,
                  exp_name, setting, model_name):
    yt = oof_df["y_true"].values.astype(int)
    yp = oof_df["y_pred"].values.astype(int)
    yb = oof_df[["prob_class_0",
                  "prob_class_1",
                  "prob_class_2"]].values.astype(float)
    m  = compute_final_metrics(yt, yp, yb)
    m.update(dict(cohort=cohort, experiment_family=exp_family,
                  experiment_name=exp_name, setting=setting,
                  model=model_name,
                  n_total_oof_samples=len(oof_df)))
    return m

# ============================================================
# 17) CV EXPERIMENT RUNNERS
#
#   Each runner receives `fold_cache` (list of 5 pre-computed dicts)
#   instead of raw arrays.  No imputation happens inside runners.
#   UDS-only runners skip MRI arrays entirely.
# ============================================================

def _log(yt, yp, yb, tag):
    fm = compute_final_metrics(yt, yp, yb)
    print(f"{tag} "
          f"acc={fm['accuracy']:.4f} "
          f"auc_macro={fm['auc_macro_ovr']:.4f} "
          f"auc_bin={fm['auc_binary_collapse_debug']:.4f}")


# ---------- UNIMODAL NON-GRAPH ----------

def run_cv_unimodal_non_graph(model_name, modality_name,
                               fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(100 + fid)
        t0  = time.time()

        if modality_name == "UDS":
            Xtr, Xva, Xte = fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"]
        else:
            Xtr, Xva, Xte = fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"]

        yt, yp, yb = run_non_graph_model(
            model_name, Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "UNIMODAL", "UNIMODAL_NON_GRAPH",
            modality_name, model_name))
        _log(yt, yp, yb,
             f"[UNI][NON_GRAPH][{modality_name}][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "UNIMODAL",
                            "UNIMODAL_NON_GRAPH", modality_name,
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- UNIMODAL GRAPH ----------

def run_cv_unimodal_graph(model_name, modality_name,
                           fold_cache, cv_splits,
                           adjacency, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(200 + fid)
        t0  = time.time()

        if modality_name == "UDS":
            Xtr, Xva, Xte = fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"]
        else:
            Xtr, Xva, Xte = fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"]

        tr_g, va_g, te_g = build_graphs_from_fold(
            Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"], adjacency)
        yt, yp, yb = run_graph_model(
            model_name, tr_g, va_g, te_g,
            in_channels=1, y_tr=fd["y_tr"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "UNIMODAL", "UNIMODAL_GRAPH",
            f"{modality_name}_{structure_name}", model_name))
        _log(yt, yp, yb,
             f"[UNI][GRAPH][{modality_name}][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "UNIMODAL",
                            "UNIMODAL_GRAPH",
                            f"{modality_name}_{structure_name}",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- EARLY FUSION NON-GRAPH ----------

def run_cv_early_fusion_non_graph(model_name, fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(300 + fid)
        t0  = time.time()

        Xtr = np.concatenate([fd["X_uds_tr"], fd["X_mri_tr"]], axis=1)
        Xva = np.concatenate([fd["X_uds_va"], fd["X_mri_va"]], axis=1)
        Xte = np.concatenate([fd["X_uds_te"], fd["X_mri_te"]], axis=1)

        yt, yp, yb = run_non_graph_model(
            model_name, Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "FUSION", "EARLY_FUSION_NON_GRAPH",
            "early", model_name))
        _log(yt, yp, yb,
             f"[EARLY_FUSION][NON_GRAPH][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "EARLY_FUSION_NON_GRAPH", "early",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- LATE FUSION NON-GRAPH ----------

def run_cv_late_fusion_non_graph(model_name, fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(400 + fid)
        t0  = time.time()

        _, _, p_uds = run_non_graph_model(
            model_name,
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"])
        _, _, p_mri = run_non_graph_model(
            model_name,
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"])

        prob = 0.5 * (p_uds + p_mri)
        pred = np.argmax(prob, axis=1)

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], fd["y_te"], pred, prob, fid,
            "ALIGNED", "FUSION", "LATE_FUSION_NON_GRAPH",
            "late", model_name))
        _log(fd["y_te"], pred, prob,
             f"[LATE_FUSION][NON_GRAPH][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "LATE_FUSION_NON_GRAPH", "late",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- EARLY FUSION GRAPH ----------

def run_cv_early_fusion_graph(model_name, fold_cache, cv_splits,
                               uds_adj, mri_adj, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(500 + fid)
        t0  = time.time()

        tr_g, va_g, te_g = build_early_fusion_graphs(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"],
            uds_adj, mri_adj)
        yt, yp, yb = run_graph_model(
            model_name, tr_g, va_g, te_g,
            in_channels=1, y_tr=fd["y_tr"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "FUSION", "EARLY_FUSION_GRAPH",
            f"early_{structure_name}", model_name))
        _log(yt, yp, yb,
             f"[EARLY_FUSION][GRAPH][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "EARLY_FUSION_GRAPH",
                            f"early_{structure_name}", model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- LATE FUSION GRAPH ----------

def run_cv_late_fusion_graph(model_name, fold_cache, cv_splits,
                              uds_adj, mri_adj, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(600 + fid)
        t0  = time.time()

        tr_ug, va_ug, te_ug = build_graphs_from_fold(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], uds_adj)
        tr_mg, va_mg, te_mg = build_graphs_from_fold(
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], mri_adj)

        _, _, p_uds = run_graph_model(
            model_name, tr_ug, va_ug, te_ug,
            in_channels=1, y_tr=fd["y_tr"])
        _, _, p_mri = run_graph_model(
            model_name, tr_mg, va_mg, te_mg,
            in_channels=1, y_tr=fd["y_tr"])

        prob = 0.5 * (p_uds + p_mri)
        pred = np.argmax(prob, axis=1)

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], fd["y_te"], pred, prob, fid,
            "ALIGNED", "FUSION", "LATE_FUSION_GRAPH",
            f"late_{structure_name}", model_name))
        _log(fd["y_te"], pred, prob,
             f"[LATE_FUSION][GRAPH][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "LATE_FUSION_GRAPH",
                            f"late_{structure_name}", model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- PROPOSED FUSION ----------

def _get_proposed_adjs(setting, uds_s, mri_s, fid):
    if setting == "structured":
        return uds_s, mri_s
    if setting == "previous_unstructured":
        return build_identity_adj(uds_s.shape[0]), \
               build_identity_adj(mri_s.shape[0])
    if setting == "new_unstructured_bad":
        return build_bad_unstructured_adj(uds_s, 1000 + fid), \
               build_bad_unstructured_adj(mri_s, 2000 + fid)
    raise ValueError(setting)


def run_cv_proposed_fusion(setting_name,
                            fold_cache, cv_splits,
                            uds_struct_adj, mri_struct_adj):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(700 + fid)
        t0  = time.time()

        uds_adj, mri_adj = _get_proposed_adjs(
            setting_name, uds_struct_adj, mri_struct_adj, fid)

        tr_ug, va_ug, te_ug = build_graphs_from_fold(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], uds_adj)
        tr_mg, va_mg, te_mg = build_graphs_from_fold(
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], mri_adj)

        yt, yp, yb = run_proposed_fusion(
            tr_ug, va_ug, te_ug,
            tr_mg, va_mg, te_mg,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        mname = f"ImprovedGTN_MetaGraphClassifier_{setting_name}"
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "PROPOSED", "PROPOSED_FUSION",
            setting_name, mname))
        _log(yt, yp, yb,
             f"[PROPOSED][{setting_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    mname   = f"ImprovedGTN_MetaGraphClassifier_{setting_name}"
    summary = summarize_oof(oof_df, "ALIGNED", "PROPOSED",
                            "PROPOSED_FUSION", setting_name, mname)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary

# ============================================================
# 18) MAIN
# ============================================================

def main():
    all_oof, all_summary = [], []

    # ----------------------------------------------------------
    # LOAD — all 183K+ rows, -999 sentinel intact in MRI
    # ----------------------------------------------------------
    data        = load_v3_data(DATA_DIR)
    y           = data["y"]
    ids         = data["ids"]
    X_uds       = data["uds_arr"]
    X_mri_raw   = data["mri_arr_raw"]
    uds_widths  = data["uds_widths"]
    n_mri_nodes = data["n_mri_nodes"]

    print(f"\n[MAIN] N={len(y)}  "
          f"UDS={X_uds.shape[1]}  MRI={X_mri_raw.shape[1]}\n")

    # ----------------------------------------------------------
    # ADJACENCY
    # ----------------------------------------------------------
    uds_adj_s = build_uds_structured_adj(uds_widths)
    uds_adj_u = build_identity_adj(X_uds.shape[1])
    mri_adj_s = build_mri_structured_adj(n_mri_nodes, MRI_ADJ_PATH)
    mri_adj_u = build_identity_adj(n_mri_nodes)

    # ----------------------------------------------------------
    # CV SPLITS  (full dataset, stratified)
    # ----------------------------------------------------------
    cv_splits = generate_cv_splits(y, n_splits=OUTER_N_SPLITS,
                                   base_seed=123)

    # ----------------------------------------------------------
    # PRE-COMPUTE FOLD ARRAYS  (5 median fits, done once)
    # ----------------------------------------------------------
    print("\n[CACHE] Pre-computing fold arrays (MICE imputation)...")
    fold_cache = precompute_fold_arrays(
        X_uds, X_mri_raw, y, ids, cv_splits)
    print("[CACHE] Done.\n")

    # ----------------------------------------------------------
    # NON-GRAPH UNIMODAL
    # ----------------------------------------------------------
    non_graph_models = ["LogisticRegression", "MLP",
                        "XGBoost", "CNN", "Transformer"]

    for mn in non_graph_models:
        if mn == "XGBoost" and not XGB_AVAILABLE:
            continue
        for mod in ("UDS", "MRI"):
            oof, s = run_cv_unimodal_non_graph(
                mn, mod, fold_cache, cv_splits)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # GRAPH UNIMODAL
    # ----------------------------------------------------------
    graph_models = ["GCN", "GAT"]

    for gm in graph_models:
        for sname, adj in [("structured",   uds_adj_s),
                            ("unstructured", uds_adj_u)]:
            oof, s = run_cv_unimodal_graph(
                gm, "UDS", fold_cache, cv_splits, adj, sname)
            all_oof.append(oof); all_summary.append(s)

        for sname, adj in [("structured",   mri_adj_s),
                            ("unstructured", mri_adj_u)]:
            oof, s = run_cv_unimodal_graph(
                gm, "MRI", fold_cache, cv_splits, adj, sname)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # NON-GRAPH FUSION
    # ----------------------------------------------------------
    for mn in non_graph_models:
        if mn == "XGBoost" and not XGB_AVAILABLE:
            continue
        oof, s = run_cv_early_fusion_non_graph(
            mn, fold_cache, cv_splits)
        all_oof.append(oof); all_summary.append(s)

        oof, s = run_cv_late_fusion_non_graph(
            mn, fold_cache, cv_splits)
        all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # GRAPH FUSION
    # ----------------------------------------------------------
    for gm in graph_models:
        for sname, ua, ma in [
                ("structured",   uds_adj_s, mri_adj_s),
                ("unstructured", uds_adj_u, mri_adj_u)]:
            oof, s = run_cv_early_fusion_graph(
                gm, fold_cache, cv_splits, ua, ma, sname)
            all_oof.append(oof); all_summary.append(s)

            oof, s = run_cv_late_fusion_graph(
                gm, fold_cache, cv_splits, ua, ma, sname)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # PROPOSED FUSION
    # ----------------------------------------------------------
    for setting in ["structured",
                    "previous_unstructured",
                    "new_unstructured_bad"]:
        oof, s = run_cv_proposed_fusion(
            setting, fold_cache, cv_splits,
            uds_adj_s, mri_adj_s)
        all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # SAVE
    # ----------------------------------------------------------
    oof_all    = pd.concat(all_oof, axis=0, ignore_index=True)
    summary_df = pd.DataFrame(all_summary)

    col_order = [
        "cohort", "experiment_family", "experiment_name", "setting",
        "model", "n_total_oof_samples",
        "accuracy", "auc_macro_ovr", "auc_binary_collapse_debug",
        "sensitivity_macro", "specificity_macro",
        "sensitivity_at_spec80_binary",
        "achieved_specificity_at_spec80_binary",
        "threshold_for_spec80_binary",
        "avg_fold_time_seconds",
    ]
    summary_df = summary_df[col_order]

    print("\n" + "=" * 120)
    print("FINAL OOF SUMMARY")
    print("=" * 120)
    print(summary_df.to_string(index=False))

    oof_all.to_csv("multiclass_5fold_oof_predictions.csv", index=False)
    summary_df.to_csv("multiclass_5fold_final_summary.csv", index=False)
    print("\nSaved: multiclass_5fold_oof_predictions.csv")
    print("       multiclass_5fold_final_summary.csv")


if __name__ == "__main__":
    main()

Using device: cpu
[DATA] total rows           : 185831
[DATA] MRI-missing rows     : 183452  (98.7 %)
[DATA] fully-aligned rows   : 2379
[DATA] UDS dim              : 134  (history=66, survey=39, testing=29)
[DATA] MRI dim              : 198
[DATA] Class distribution   : {0: 97491, 1: 32490, 2: 55850}

[MAIN] N=185831  UDS=134  MRI=198

[ADJ] Identity adjacency for MRI  (n=198)

[CACHE] Pre-computing fold arrays (MICE imputation)...
[CACHE] fold 0 imputed  train=130081  val=18583  test=37167
[CACHE] fold 1 imputed  train=130081  val=18584  test=37166
[CACHE] fold 2 imputed  train=130081  val=18584  test=37166
[CACHE] fold 3 imputed  train=130081  val=18584  test=37166
[CACHE] fold 4 imputed  train=130081  val=18584  test=37166
[CACHE] Done.

[UNI][NON_GRAPH][UDS][LogisticRegression][f=0] acc=0.7667 auc_macro=0.8942 auc_bin=0.9205
[UNI][NON_GRAPH][UDS][LogisticRegression][f=1] acc=0.7706 auc_macro=0.8965 auc_bin=0.9237
[UNI][NON_GRAPH][UDS][LogisticRegression][f=2] acc=0.7744 auc_macro=

KeyboardInterrupt: 

diagnosis - all samples - uds aware imputation

In [ ]:
# ============================================================
# COMPLETE REWRITE — v5 FAST:
#   5-FOLD STRATIFIED OUTER CV
#   + INNER STRATIFIED TRAIN/VAL SPLIT
#
# DATASET:
#   preprocessed_v3_for_joint_missingMRI_keepRawDims
#   Files:
#     subject_id.csv  — subject identifiers (row-aligned)
#     label.csv       — integer labels 0/1/2
#     uds_history.csv — already imputed + scaled
#     uds_survey.csv  — already imputed + scaled
#     uds_testing.csv — already imputed + scaled
#     mrisbm.csv      — already scaled; -999.0 sentinel = MRI missing
#
# MODALITIES:
#   UDS = concat(uds_history, uds_survey, uds_testing)  shape (N, D_uds)
#   MRI = mrisbm flat array                              shape (N, D_mri)
#   ALL rows kept — including MRI-missing rows.
#
# MISSING MRI HANDLING — UDS-INFORMED RIDGE IMPUTATION (no leakage):
#   MRI-missing rows have ALL 198 MRI columns set to -999.0.
#   UDS (134 features, always complete) is used as predictor for Ridge.
#   One multi-output Ridge(alpha=1.0) is fit on train-aligned rows only,
#   then used to predict MRI values for all missing rows in train/val/test.
#   Leakage guarantee: Ridge.fit() sees only train-aligned rows;
#   val/test are only ever passed to Ridge.predict() with frozen weights.
#
# SPEED IMPROVEMENTS vs v4:
#   1. Fold array caching  — precompute_fold_arrays() runs once before
#                            all experiments; 5 Ridge fits total instead
#                            of one per runner per fold (85×)
#   2. UDS-only skip       — UDS experiments need no imputation at all;
#                            pure numpy slice with zero extra cost
#   3. Graph batch size    — 32 → 512 (16× fewer batches per epoch)
#   4. Epochs / patience   — max_epochs 50→30, patience 10→7 everywhere
#                            GTN pretraining 50→30 epochs
#                            MetaGraphClassifier patience 20→10
#   5. MICE speed tuning   — n_nearest_features=30 limits each column
#                            regression to 30 predictors instead of 198
#
# TASK:
#   MULTICLASS DIAGNOSIS
#     class 0: LABEL 0  (original NACCUDSD in [1, 2])
#     class 1: LABEL 1  (original NACCUDSD == 3)
#     class 2: LABEL 2  (original NACCUDSD == 4)
#
# SPLIT DESIGN (PER OUTER FOLD):
#   test  = 20 % of full dataset
#   remaining 80 %:
#     val   = 12.5 % of remaining  = 10 % of total
#     train = 87.5 % of remaining  = 70 % of total
#
# EVALUATION:
#   Aggregate ALL out-of-fold predictions → compute metrics from
#   concatenated OOF array (not per-fold average).
#
# OUTPUTS:
#   multiclass_5fold_oof_predictions.csv
#   multiclass_5fold_final_summary.csv
# ============================================================

import os
import time
import copy
import random
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import Ridge
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader

from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import (
    GCNConv,
    GATConv,
    TransformerConv,
    global_mean_pool,
    global_max_pool,
)

warnings.filterwarnings("ignore")

# ============================================================
# 0) OPTIONAL XGBOOST
# ============================================================

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False
    print("WARNING: xgboost not installed — XGBoost runs will be skipped.")

# ============================================================
# 1) GLOBAL CONSTANTS / DEVICE
# ============================================================

NUM_CLASSES    = 3
OUTER_N_SPLITS = 5
MISSING_VALUE  = -999.0

DATA_DIR     = "./preprocessed_v3_for_joint_missingMRI_keepRawDims"
MRI_ADJ_PATH = "./combined_adjacency_matrix.csv"

# Training hyper-parameters
GRAPH_BATCH_SIZE = 512
TAB_BATCH_SIZE   = 64
MAX_EPOCHS       = 30
PATIENCE         = 7
GTN_EPOCHS       = 30
META_PATIENCE    = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2) CLASS WEIGHT HELPERS
# ============================================================

def compute_class_weight_vector(y: np.ndarray,
                                num_classes: int = NUM_CLASSES) -> np.ndarray:
    y      = np.asarray(y).astype(int)
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    total  = counts.sum()
    w = np.zeros(num_classes, dtype=np.float32)
    for c in range(num_classes):
        if counts[c] > 0:
            w[c] = float(total / (num_classes * counts[c]))
    return w


def compute_class_weight_dict(y: np.ndarray,
                               num_classes: int = NUM_CLASSES
                               ) -> Dict[int, float]:
    v = compute_class_weight_vector(y, num_classes)
    return {c: float(v[c]) for c in range(num_classes) if v[c] > 0}


def compute_sample_weights(y: np.ndarray,
                           num_classes: int = NUM_CLASSES) -> np.ndarray:
    v = compute_class_weight_vector(y, num_classes)
    return v[np.asarray(y).astype(int)].astype(np.float32)


def class_weight_tensor(y: np.ndarray,
                        device=DEVICE,
                        num_classes: int = NUM_CLASSES) -> torch.Tensor:
    v = compute_class_weight_vector(y, num_classes)
    return torch.tensor(v, dtype=torch.float32, device=device)

# ============================================================
# 3) DATA LOADING
#    Keeps ALL rows including MRI-missing.
#    MRI-missing rows retain -999.0 — imputation happens
#    inside the pre-computed fold cache (Section 8).
# ============================================================

def load_v3_data(data_dir: str = DATA_DIR) -> dict:
    """
    Returns:
      ids          — subject IDs,          shape (N,)
      y            — integer labels 0/1/2, shape (N,)
      uds_arr      — float32 UDS,          shape (N, D_uds)
      mri_arr_raw  — float32 MRI,          shape (N, D_mri)
                     -999.0 where MRI was absent
      mri_missing  — bool mask,            shape (N,)
      uds_widths   — dict {history, survey, testing} column counts
      n_mri_nodes  — D_mri
    """
    subject_df = pd.read_csv(os.path.join(data_dir, "subject_id.csv"))
    label_df   = pd.read_csv(os.path.join(data_dir, "label.csv"))
    history_df = pd.read_csv(os.path.join(data_dir, "uds_history.csv"))
    survey_df  = pd.read_csv(os.path.join(data_dir, "uds_survey.csv"))
    testing_df = pd.read_csv(os.path.join(data_dir, "uds_testing.csv"))
    mri_df     = pd.read_csv(os.path.join(data_dir, "mrisbm.csv"))

    n = len(label_df)
    assert (len(subject_df) == len(history_df) == len(survey_df) ==
            len(testing_df) == len(mri_df) == n), \
        "Row-count mismatch across preprocessed files."

    id_col = subject_df.columns[0]
    ids    = subject_df[id_col].values
    y      = label_df["LABEL"].astype(int).values

    history_arr = history_df.values.astype(np.float32)
    survey_arr  = survey_df.values.astype(np.float32)
    testing_arr = testing_df.values.astype(np.float32)
    uds_arr     = np.concatenate(
        [history_arr, survey_arr, testing_arr], axis=1)

    uds_widths = {
        "history": history_arr.shape[1],
        "survey":  survey_arr.shape[1],
        "testing": testing_arr.shape[1],
    }

    mri_arr_raw = mri_df.values.astype(np.float32)
    mri_missing = np.all(mri_arr_raw == MISSING_VALUE, axis=1)

    n_miss = int(mri_missing.sum())
    print(f"[DATA] total rows           : {n}")
    print(f"[DATA] MRI-missing rows     : {n_miss}  "
          f"({100*n_miss/n:.1f} %)")
    print(f"[DATA] fully-aligned rows   : {n - n_miss}")
    print(f"[DATA] UDS dim              : {uds_arr.shape[1]}  "
          f"(history={uds_widths['history']}, "
          f"survey={uds_widths['survey']}, "
          f"testing={uds_widths['testing']})")
    print(f"[DATA] MRI dim              : {mri_arr_raw.shape[1]}")
    print(f"[DATA] Class distribution   : "
          f"{ {c: int((y==c).sum()) for c in range(NUM_CLASSES)} }")

    return dict(ids=ids, y=y, uds_arr=uds_arr,
                mri_arr_raw=mri_arr_raw, mri_missing=mri_missing,
                uds_widths=uds_widths,
                n_mri_nodes=int(mri_arr_raw.shape[1]))

# ============================================================
# 4) ADJACENCY BUILDERS
# ============================================================

def build_uds_structured_adj(uds_widths: dict) -> np.ndarray:
    """Intra-block fully-connected; history / survey / testing blocks."""
    n_hist = uds_widths["history"]
    n_surv = uds_widths["survey"]
    n_test = uds_widths["testing"]
    total  = n_hist + n_surv + n_test
    adj    = np.zeros((total, total), dtype=np.float32)
    for off, sz in zip([0, n_hist, n_hist + n_surv],
                       [n_hist, n_surv, n_test]):
        adj[off:off+sz, off:off+sz] = 1.0
    np.fill_diagonal(adj, 1.0)
    return adj


def build_mri_structured_adj(n_nodes: int,
                              adj_path: str = MRI_ADJ_PATH) -> np.ndarray:
    if os.path.exists(adj_path):
        try:
            adj = pd.read_csv(adj_path, index_col=0,
                              header=0).values.astype(float)
            if (adj.shape == (n_nodes, n_nodes) and
                    np.allclose(adj, adj.T)):
                print(f"[ADJ] Loaded MRI adjacency  shape={adj.shape}")
                return adj.astype(np.float32)
            print("[ADJ] MRI adj wrong shape/asymmetric — using identity.")
        except Exception as exc:
            print(f"[ADJ] Could not load adj: {exc} — using identity.")
    print(f"[ADJ] Identity adjacency for MRI  (n={n_nodes})")
    return np.eye(n_nodes, dtype=np.float32)


def build_identity_adj(n: int) -> np.ndarray:
    return np.eye(n, dtype=np.float32)


def build_bad_unstructured_adj(structured_adj: np.ndarray,
                                seed: int) -> np.ndarray:
    rng    = np.random.default_rng(seed)
    n      = structured_adj.shape[0]
    out    = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(out, 1.0)
    upper  = np.triu(structured_adj, k=1)
    n_edge = int((upper > 0).sum())
    ai, aj = np.triu_indices(n, k=1)
    ch     = rng.choice(len(ai), size=n_edge, replace=False)
    out[ai[ch], aj[ch]] = 1.0
    out[aj[ch], ai[ch]] = 1.0
    return out


def adjacency_to_edge_index(adj: np.ndarray) -> torch.Tensor:
    ei = np.array(np.nonzero(adj), dtype=np.int64)
    return torch.tensor(ei, dtype=torch.long)


def block_diag_adjacency(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    na, nb = a.shape[0], b.shape[0]
    out = np.zeros((na + nb, na + nb), dtype=float)
    out[:na, :na] = a
    out[na:, na:] = b
    return out

# ============================================================
# 5) METRICS
# ============================================================

def compute_multiclass_specificity(y_true, y_pred, nc=3):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(nc)))
    s  = []
    for c in range(nc):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - (cm[c, :].sum() - tp) - fp
        s.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return float(np.nanmean(s))


def compute_multiclass_sensitivity(y_true, y_pred, nc=3):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(nc)))
    s  = []
    for c in range(nc):
        tp = cm[c, c]; fn = cm[c, :].sum() - tp
        s.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
    return float(np.nanmean(s))


def compute_multiclass_auc(y_true, y_prob):
    try:
        yb = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        return float(roc_auc_score(yb, y_prob,
                                   multi_class="ovr", average="macro"))
    except Exception:
        return np.nan


def compute_binary_auc(y_true, y_prob):
    try:
        yb  = (np.asarray(y_true) != 0).astype(int)
        pp  = np.asarray(y_prob)[:, 1] + np.asarray(y_prob)[:, 2]
        return float(roc_auc_score(yb, pp)) \
            if len(np.unique(yb)) == 2 else np.nan
    except Exception:
        return np.nan


def sensitivity_at_80_spec(y_true, y_prob):
    try:
        yb  = (np.asarray(y_true) != 0).astype(int)
        sc  = np.asarray(y_prob)[:, 1] + np.asarray(y_prob)[:, 2]
        if len(np.unique(yb)) < 2:
            return dict(threshold=np.nan,
                        sensitivity=np.nan, specificity=np.nan)
        fpr, tpr, thr = roc_curve(yb, sc)
        spec = 1.0 - fpr
        idx  = int(np.argmin(np.abs(spec - 0.80)))
        return dict(threshold=float(thr[idx]),
                    sensitivity=float(tpr[idx]),
                    specificity=float(spec[idx]))
    except Exception:
        return dict(threshold=np.nan,
                    sensitivity=np.nan, specificity=np.nan)


def compute_final_metrics(y_true, y_pred, y_prob):
    s80 = sensitivity_at_80_spec(y_true, y_prob)
    return {
        "accuracy":                              accuracy_score(y_true, y_pred),
        "auc_macro_ovr":                         compute_multiclass_auc(y_true, y_prob),
        "auc_binary_collapse_debug":             compute_binary_auc(y_true, y_prob),
        "sensitivity_macro":                     compute_multiclass_sensitivity(y_true, y_pred, NUM_CLASSES),
        "specificity_macro":                     compute_multiclass_specificity(y_true, y_pred, NUM_CLASSES),
        "sensitivity_at_spec80_binary":          s80["sensitivity"],
        "achieved_specificity_at_spec80_binary": s80["specificity"],
        "threshold_for_spec80_binary":           s80["threshold"],
    }

# ============================================================
# 6) CV SPLITS
# ============================================================

def generate_cv_splits(y: np.ndarray,
                       n_splits: int = 5,
                       base_seed: int = 123) -> List[dict]:
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True,
                           random_state=base_seed)
    full = np.arange(len(y))
    out  = []
    for fold_id, (outer_tr, test_idx) in enumerate(skf.split(full, y)):
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=0.125,
            random_state=base_seed + fold_id)
        inner_tr, val_idx = next(sss.split(outer_tr, y[outer_tr]))
        out.append(dict(fold=fold_id,
                        train_idx=outer_tr[inner_tr],
                        val_idx=outer_tr[val_idx],
                        test_idx=test_idx))
    return out

# ============================================================
# 7) UDS-INFORMED RIDGE IMPUTATION (MRI only, no leakage)
#
#   Core idea:
#     Every row — including MRI-missing ones — has COMPLETE UDS
#     features (134 columns, already imputed + scaled).  UDS and
#     MRI measure the same patient, sharing latent structure
#     (diagnosis, age, cognitive profile, neurological state).
#     We exploit this by fitting one Ridge regression model that
#     predicts ALL MRI columns simultaneously from UDS.
#
#   Why this outperforms KNN and MICE for this dataset:
#     - KNN on fully-missing MRI rows: all 198 MRI columns are
#       absent, so distance computation falls back to mean-imputed
#       space where rows are nearly identical.  "distance" weights
#       become meaningless.  Result ≈ column means.
#     - MICE on fully-missing rows: iterates over MRI-MRI cross-
#       column regressions, but all initial values for missing rows
#       are mean-filled, and they stay similar across iterations
#       because the only information is from the 1.3% complete rows.
#     - Ridge uses UDS as the predictor (always complete, 134 feats).
#       UDS↔MRI correlations are strong: neuropsychological test
#       scores, functional assessments, and demographic features
#       all reflect the same underlying neurology as brain volume
#       and cortical thickness.  This yields genuinely patient-
#       specific MRI imputations.
#
#   Procedure (leakage-free at every step):
#     1. Identify aligned rows (MRI present) in TRAIN split only.
#     2. Fit Ridge(alpha=1.0) on
#            X = UDS_train[aligned],  Y = MRI_train[aligned]
#        Multi-output Ridge solves all D_mri columns in one step.
#        Only train-aligned rows are used → no val/test statistics.
#     3. For MRI-missing rows in each split:
#            imputed_MRI[i] = ridge.predict(UDS[i])
#     4. For MRI-present rows: keep original MRI values unchanged.
#     5. Safety clip: replace any residual sentinel / nan with 0.0.
#
#   Leakage guarantee (three independent levels):
#     - Ridge.fit() sees ONLY train-split rows that have real MRI.
#     - Ridge regression coefficients are frozen after fit().
#     - Val/test UDS rows are only passed to Ridge.predict(),
#       which is a pure read of the already-learned weight matrix.
#     - The aligned-row mask is derived from train MRI only.
#
#   Speed:
#     Ridge on ~1.6K aligned train rows × 134 UDS features,
#     predicting 198 MRI columns ≈ instant (< 1 s per fold).
#     No O(N²) distance computation; no iterative convergence loop.
# ============================================================

RIDGE_ALPHA = 1.0


def impute_mri_fold(mri_train: np.ndarray,
                    mri_val:   np.ndarray,
                    mri_test:  np.ndarray,
                    uds_train: np.ndarray,
                    uds_val:   np.ndarray,
                    uds_test:  np.ndarray,
                    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    UDS-informed Ridge imputation for one CV fold.

    Args:
        mri_train / mri_val / mri_test  — raw MRI arrays;
            rows where MRI was absent are filled with MISSING_VALUE.
        uds_train / uds_val / uds_test  — corresponding UDS arrays;
            always fully present (no sentinel values).

    Returns:
        Three float32 MRI arrays.  MRI-present rows are unchanged.
        MRI-missing rows are filled with Ridge predictions from UDS.

    Leakage proof:
        Ridge.fit() is called exactly once on train-aligned rows.
        Val and test only ever call Ridge.predict() with the frozen
        coefficient matrix — no statistics are updated.
    """
    D_mri = mri_train.shape[1]

    # Step 1 — identify aligned (MRI-present) rows in TRAIN only
    train_missing_mask = np.all(mri_train == MISSING_VALUE, axis=1)
    train_aligned_mask = ~train_missing_mask   # True = MRI present

    # Safety fallback: if train has zero aligned MRI rows (degenerate
    # fold), fill all missing rows with zeros and return.
    if train_aligned_mask.sum() == 0:
        def _zero_fill(mri):
            out  = mri.copy().astype(np.float32)
            miss = np.all(out == MISSING_VALUE, axis=1)
            out[miss] = 0.0
            out[out == MISSING_VALUE] = 0.0
            return out
        return _zero_fill(mri_train), _zero_fill(mri_val), _zero_fill(mri_test)

    # Step 2 — fit multi-output Ridge on TRAIN aligned rows only
    #   Ridge handles multi-output via one Cholesky solve, which is
    #   O(D_uds² × D_mri) — fast even for D_mri=198.
    X_fit = uds_train[train_aligned_mask].astype(np.float64)
    Y_fit = mri_train[train_aligned_mask].astype(np.float64)

    ridge = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    ridge.fit(X_fit, Y_fit)   # ← only train-aligned rows seen here

    # Steps 3 & 4 — impute missing rows, preserve present rows
    def _apply(mri: np.ndarray, uds: np.ndarray) -> np.ndarray:
        out          = mri.copy().astype(np.float32)
        missing_mask = np.all(out == MISSING_VALUE, axis=1)

        if missing_mask.sum() > 0:
            # Ridge.predict() uses frozen coefficients — read-only
            X_pred            = uds[missing_mask].astype(np.float64)
            predicted         = ridge.predict(X_pred).astype(np.float32)
            out[missing_mask] = predicted

        # Step 5 — safety: clear any residual sentinel / nan
        out = np.where(out == MISSING_VALUE, 0.0, out).astype(np.float32)
        out = np.nan_to_num(out, nan=0.0)
        return out.astype(np.float32)

    return _apply(mri_train, uds_train),            _apply(mri_val,   uds_val),              _apply(mri_test,  uds_test)

# ============================================================
# 8) FOLD ARRAY CACHE
#
#   Runs ONCE before all experiments.
#   Produces 5 fold dicts, each containing:
#     X_uds_{tr,va,te}  — UDS slices (no imputation needed)
#     X_mri_{tr,va,te}  — MRI slices, MICE-imputed (train stats only)
#     y_{tr,va,te}       — label arrays
#     ids_te             — subject IDs for OOF rows
#
#   Every CV runner receives the pre-computed list and indexes by fold_id.
#   Total imputation cost: 5 fits instead of one per runner × per fold.
# ============================================================

def precompute_fold_arrays(X_uds:      np.ndarray,
                           X_mri_raw:  np.ndarray,
                           y:          np.ndarray,
                           ids:        np.ndarray,
                           cv_splits:  List[dict]) -> List[dict]:
    """
    Pre-impute MRI for every fold.  UDS is sliced without processing.
    """
    cache = []
    for split in cv_splits:
        fold_id = split["fold"]
        ti, vi, tei = (split["train_idx"],
                       split["val_idx"],
                       split["test_idx"])

        # UDS — pure slice
        X_uds_tr = X_uds[ti].astype(np.float32)
        X_uds_va = X_uds[vi].astype(np.float32)
        X_uds_te = X_uds[tei].astype(np.float32)

        # MRI — UDS-informed Ridge impute, fit on train-aligned rows only
        X_mri_tr, X_mri_va, X_mri_te = impute_mri_fold(
            mri_train=X_mri_raw[ti],
            mri_val=X_mri_raw[vi],
            mri_test=X_mri_raw[tei],
            uds_train=X_uds_tr,   # always complete; used as predictors
            uds_val=X_uds_va,
            uds_test=X_uds_te,
        )

        cache.append(dict(
            fold=fold_id,
            X_uds_tr=X_uds_tr, X_uds_va=X_uds_va, X_uds_te=X_uds_te,
            X_mri_tr=X_mri_tr, X_mri_va=X_mri_va, X_mri_te=X_mri_te,
            y_tr=y[ti], y_va=y[vi], y_te=y[tei],
            ids_te=ids[tei],
        ))
        print(f"[CACHE] fold {fold_id} imputed  "
              f"train={len(ti)}  val={len(vi)}  test={len(tei)}")

    return cache

# ============================================================
# 9) TORCH DATASETS
# ============================================================

class ArrayDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================================
# 10) NON-GRAPH MODELS
# ============================================================

class MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128),
                 dropout=0.2, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN1DNet(nn.Module):
    def __init__(self, seq_len, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.classifier(
            self.features(x.unsqueeze(1)).squeeze(-1))


class TabularTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128,
                 dropout=0.1, num_classes=3):
        super().__init__()
        self.proj    = nn.Linear(1, d_model)
        enc          = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="relu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.cls     = nn.Linear(d_model, num_classes)

    def forward(self, x):
        return self.cls(
            self.encoder(self.proj(x.unsqueeze(-1))).mean(dim=1))

# ============================================================
# 11) GRAPH MODELS
# ============================================================

class GCNGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels=64,
                 dropout=0.2, num_classes=3):
        super().__init__()
        self.conv1   = GCNConv(in_channels, hidden_channels)
        self.conv2   = GCNConv(hidden_channels, hidden_channels)
        self.lin     = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        return self.lin(global_mean_pool(x, b))


class GATGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels=32,
                 heads=4, dropout=0.2, num_classes=3):
        super().__init__()
        self.gat1    = GATConv(in_channels, hidden_channels,
                               heads=heads, dropout=dropout)
        self.gat2    = GATConv(hidden_channels * heads, hidden_channels,
                               heads=1, dropout=dropout)
        self.lin     = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.gat1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.gat2(x, ei))
        return self.lin(global_mean_pool(x, b))

# ============================================================
# 12) TRAINING / INFERENCE HELPERS
# ============================================================

def _train_torch(model, train_loader, val_loader, cw,
                 lr=1e-3, wd=1e-4,
                 max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model     = model.to(DEVICE)
    cw        = cw.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_loss, best_state, pat = float("inf"), None, 0

    for _ in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl.append(criterion(model(xb.to(DEVICE)),
                                    yb.to(DEVICE)).item())
        v = float(np.mean(vl))
        if v < best_loss:
            best_loss  = v
            best_state = {k: w.cpu().clone()
                          for k, w in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def _predict_torch(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            lg = model(xb.to(DEVICE))
            probs.extend(torch.softmax(lg, 1).cpu().numpy())
            preds.extend(torch.argmax(lg, 1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(trues), np.array(preds), np.array(probs)


def _train_pyg(model, train_loader, val_loader, cw,
               lr=1e-3, wd=1e-4,
               max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model     = model.to(DEVICE)
    cw        = cw.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_loss, best_state, pat = float("inf"), None, 0

    for _ in range(max_epochs):
        model.train()
        for b in train_loader:
            b = b.to(DEVICE)
            opt.zero_grad()
            criterion(model(b), b.y.view(-1)).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(DEVICE)
                vl.append(criterion(model(b),
                                    b.y.view(-1)).item())
        v = float(np.mean(vl))
        if v < best_loss:
            best_loss  = v
            best_state = {k: w.cpu().clone()
                          for k, w in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def _predict_pyg(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for b in loader:
            b  = b.to(DEVICE)
            lg = model(b)
            probs.extend(torch.softmax(lg, 1).cpu().numpy())
            preds.extend(torch.argmax(lg, 1).cpu().numpy())
            trues.extend(b.y.view(-1).cpu().numpy())
    return np.array(trues), np.array(preds), np.array(probs)

# ============================================================
# 13) GRAPH CONSTRUCTION
#     Each feature column = 1 node with 1 feature.
#     edge_index is built once per adjacency and reused.
#     All values are already imputed — no sentinel values here.
# ============================================================

def _make_flat_graphs(X: np.ndarray, y: np.ndarray,
                      edge_index: torch.Tensor) -> List[Data]:
    return [
        Data(x=torch.tensor(X[i].reshape(-1, 1), dtype=torch.float32),
             edge_index=edge_index,
             y=torch.tensor([int(y[i])], dtype=torch.long))
        for i in range(len(X))
    ]


def build_graphs_from_fold(X_tr, X_va, X_te,
                            y_tr, y_va, y_te,
                            adjacency: np.ndarray
                            ) -> Tuple[List, List, List]:
    ei = adjacency_to_edge_index(adjacency)
    return (_make_flat_graphs(X_tr, y_tr, ei),
            _make_flat_graphs(X_va, y_va, ei),
            _make_flat_graphs(X_te, y_te, ei))


def build_early_fusion_graphs(X_uds_tr, X_uds_va, X_uds_te,
                               X_mri_tr, X_mri_va, X_mri_te,
                               y_tr, y_va, y_te,
                               uds_adj: np.ndarray,
                               mri_adj: np.ndarray
                               ) -> Tuple[List, List, List]:
    ei = adjacency_to_edge_index(block_diag_adjacency(uds_adj, mri_adj))

    def _make(Xu, Xm, y):
        return [
            Data(x=torch.tensor(
                     np.concatenate([Xu[i], Xm[i]]).reshape(-1, 1),
                     dtype=torch.float32),
                 edge_index=ei,
                 y=torch.tensor([int(y[i])], dtype=torch.long))
            for i in range(len(y))
        ]

    return _make(X_uds_tr, X_mri_tr, y_tr), \
           _make(X_uds_va, X_mri_va, y_va), \
           _make(X_uds_te, X_mri_te, y_te)

# ============================================================
# 14) NON-GRAPH MODEL RUNNERS
# ============================================================

def _run_lr(X_tr, X_te, y_tr, y_te):
    m = LogisticRegression(
        max_iter=3000, solver="lbfgs", multi_class="multinomial",
        class_weight=compute_class_weight_dict(y_tr))
    m.fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)
    return y_te, np.argmax(prob, 1), prob


def _run_xgb(X_tr, X_te, y_tr, y_te):
    if not XGB_AVAILABLE:
        prob = np.full((len(X_te), NUM_CLASSES), np.nan)
        return y_te, np.zeros(len(X_te), dtype=int), prob
    m = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="multi:softprob", num_class=NUM_CLASSES,
        eval_metric="mlogloss", random_state=0)
    m.fit(X_tr, y_tr, sample_weight=compute_sample_weights(y_tr))
    prob = m.predict_proba(X_te)
    return y_te, np.argmax(prob, 1), prob


def _run_deep(model_name, X_tr, X_va, X_te, y_tr, y_va, y_te):
    trl = TorchDataLoader(ArrayDataset(X_tr, y_tr),
                          batch_size=TAB_BATCH_SIZE, shuffle=True)
    val = TorchDataLoader(ArrayDataset(X_va, y_va),
                          batch_size=TAB_BATCH_SIZE, shuffle=False)
    tel = TorchDataLoader(ArrayDataset(X_te, y_te),
                          batch_size=TAB_BATCH_SIZE, shuffle=False)
    d = X_tr.shape[1]
    if model_name == "MLP":
        m = MLPNet(d, num_classes=NUM_CLASSES)
    elif model_name == "CNN":
        m = CNN1DNet(d, num_classes=NUM_CLASSES)
    elif model_name == "Transformer":
        m = TabularTransformer(d, d_model=64, nhead=4,
                               num_layers=2, num_classes=NUM_CLASSES)
    else:
        raise ValueError(model_name)
    m = _train_torch(m, trl, val, class_weight_tensor(y_tr))
    return _predict_torch(m, tel)


def run_non_graph_model(model_name,
                        X_tr, X_va, X_te,
                        y_tr, y_va, y_te):
    if model_name == "LogisticRegression":
        return _run_lr(X_tr, X_te, y_tr, y_te)
    if model_name == "XGBoost":
        return _run_xgb(X_tr, X_te, y_tr, y_te)
    if model_name in ("MLP", "CNN", "Transformer"):
        return _run_deep(model_name, X_tr, X_va, X_te,
                         y_tr, y_va, y_te)
    raise ValueError(f"Unknown model: {model_name}")


def run_graph_model(model_name, tr_g, va_g, te_g,
                    in_channels: int, y_tr: np.ndarray):
    trl = PyGDataLoader(tr_g, batch_size=GRAPH_BATCH_SIZE, shuffle=True)
    val = PyGDataLoader(va_g, batch_size=GRAPH_BATCH_SIZE, shuffle=False)
    tel = PyGDataLoader(te_g, batch_size=GRAPH_BATCH_SIZE, shuffle=False)

    if model_name == "GCN":
        m = GCNGraphClassifier(in_channels, 64, num_classes=NUM_CLASSES)
    elif model_name == "GAT":
        m = GATGraphClassifier(in_channels, 32, heads=4,
                               num_classes=NUM_CLASSES)
    else:
        raise ValueError(model_name)

    m = _train_pyg(m, trl, val, class_weight_tensor(y_tr))
    return _predict_pyg(m, tel)

# ============================================================
# 15) PROPOSED MODEL — GTN pretraining + MetaGraphClassifier
# ============================================================

class ImprovedGTN(nn.Module):
    def __init__(self, num_node_features, hidden_channels,
                 num_classes, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(num_node_features,
                                     hidden_channels, heads=heads,
                                     dropout=0.1)
        self.conv2 = TransformerConv(hidden_channels * heads,
                                     hidden_channels, heads=heads,
                                     dropout=0.1)
        self.conv3 = TransformerConv(hidden_channels * heads,
                                     32, heads=1, dropout=0.1)
        self.bn    = nn.LayerNorm(32)
        self.fc    = nn.Linear(32, num_classes)
        self.drop  = nn.Dropout(p=0.3)

    def forward(self, data, return_edge_weights=False):
        x, ei = data.x, data.edge_index
        batch  = (data.batch if hasattr(data, "batch")
                  else torch.zeros(x.size(0), dtype=torch.long,
                                   device=x.device))
        x = F.relu(self.conv1(x, ei)); x = self.drop(x)
        x = F.relu(self.conv2(x, ei)); x = self.drop(x)

        if return_edge_weights:
            x, (ei_out, ew_out) = self.conv3(
                x, ei, return_attention_weights=True)
            ei_out = ei_out.to(x.device)
            ew_out = ew_out.to(x.device)
        else:
            x = self.conv3(x, ei)

        x      = self.bn(F.relu(x))
        pooled = global_max_pool(x, batch)
        if return_edge_weights:
            return pooled, x, ei_out, ew_out
        return self.fc(pooled)


class MetaGraphClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = TransformerConv(input_dim, hidden_dim,
                                     heads=2, dropout=0.1)
        self.conv2 = TransformerConv(hidden_dim * 2, hidden_dim,
                                     heads=1, dropout=0.1)
        self.cls   = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.relu(self.conv2(x, ei))
        return self.cls(global_mean_pool(x, b))


def _xavier_init(m):
    for layer in m.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


def _train_gnns(loaders, gnns, y_tr):
    cw  = class_weight_tensor(y_tr)
    crit = nn.CrossEntropyLoss(weight=cw)
    for g in gnns:
        g.to(DEVICE)
    opts = [torch.optim.Adam(g.parameters(), lr=0.002,
                             weight_decay=1e-4) for g in gnns]
    for _ in range(GTN_EPOCHS):
        for loader, gnn, opt in zip(loaders, gnns, opts):
            gnn.train()
            for b in loader:
                b = b.to(DEVICE)
                opt.zero_grad()
                crit(gnn(b), b.y.view(-1)).backward()
                opt.step()


def _extract_features(dataset, gnn):
    nf, ei, ew = [], [], []
    gnn.eval()
    with torch.no_grad():
        for d in dataset:
            d = d.to(DEVICE)
            _, n, e_i, e_w = gnn(d, return_edge_weights=True)
            nf.append(n.cpu())
            ei.append(e_i.cpu())
            ew.append(e_w.cpu())
    return nf, ei, ew


def _build_new_graphs(nf, ei, ew, y):
    out = []
    for x, e_i, e_w, label in zip(nf, ei, ew, y):
        if e_w.ndim > 1:
            e_w = e_w.mean(dim=1)
        out.append(Data(x=x, edge_index=e_i, edge_attr=e_w,
                        y=torch.tensor([int(label)],
                                       dtype=torch.long)))
    return out


def _merge_graphs(g1, g2):
    off = g1.x.size(0)
    cx  = torch.cat([g1.x, g2.x], dim=0)
    cei = torch.cat([g1.edge_index, g2.edge_index + off], dim=1)
    cea = (torch.cat([g1.edge_attr, g2.edge_attr], dim=0)
           if (getattr(g1, "edge_attr", None) is not None and
               getattr(g2, "edge_attr", None) is not None)
           else None)
    return Data(x=cx, edge_index=cei, edge_attr=cea, y=g1.y)


def _norm_graph_features(tr, va, te):
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(
        torch.cat([d.x for d in tr], 0).numpy())

    def _tx(ds):
        out = []
        for d in ds:
            dc   = copy.deepcopy(d)
            dc.x = torch.tensor(
                sc.transform(dc.x.numpy()), dtype=torch.float32)
            out.append(dc)
        return out

    return _tx(tr), _tx(va), _tx(te)


def run_proposed_fusion(tr_uds, va_uds, te_uds,
                        tr_mri, va_mri, te_mri,
                        y_tr, y_va, y_te):
    tr_uds, va_uds, te_uds = _norm_graph_features(tr_uds, va_uds, te_uds)
    tr_mri, va_mri, te_mri = _norm_graph_features(tr_mri, va_mri, te_mri)

    trl_u = PyGDataLoader(tr_uds, batch_size=GRAPH_BATCH_SIZE,
                          shuffle=False)
    trl_m = PyGDataLoader(tr_mri, batch_size=GRAPH_BATCH_SIZE,
                          shuffle=False)

    gnn_u = ImprovedGTN(1, 64, NUM_CLASSES)
    gnn_m = ImprovedGTN(1, 64, NUM_CLASSES)
    _xavier_init(gnn_u); _xavier_init(gnn_m)
    _train_gnns([trl_u, trl_m], [gnn_u, gnn_m], y_tr)

    def _feats(ds, gnn):
        return _extract_features(ds, gnn)

    tr_nu, tr_eu, tr_wu = _feats(tr_uds, gnn_u)
    va_nu, va_eu, va_wu = _feats(va_uds, gnn_u)
    te_nu, te_eu, te_wu = _feats(te_uds, gnn_u)
    tr_nm, tr_em, tr_wm = _feats(tr_mri, gnn_m)
    va_nm, va_em, va_wm = _feats(va_mri, gnn_m)
    te_nm, te_em, te_wm = _feats(te_mri, gnn_m)

    m_tr = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(tr_nu, tr_eu, tr_wu, y_tr),
        _build_new_graphs(tr_nm, tr_em, tr_wm, y_tr))]
    m_va = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(va_nu, va_eu, va_wu, y_va),
        _build_new_graphs(va_nm, va_em, va_wm, y_va))]
    m_te = [_merge_graphs(g1, g2) for g1, g2 in zip(
        _build_new_graphs(te_nu, te_eu, te_wu, y_te),
        _build_new_graphs(te_nm, te_em, te_wm, y_te))]

    tr_b  = Batch.from_data_list(m_tr).to(DEVICE)
    va_b  = Batch.from_data_list(m_va).to(DEVICE)
    te_b  = Batch.from_data_list(m_te).to(DEVICE)
    tr_lb = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    va_lb = torch.tensor(y_va, dtype=torch.long, device=DEVICE)

    cw   = class_weight_tensor(y_tr)
    crit = nn.CrossEntropyLoss(weight=cw)
    meta = MetaGraphClassifier(32, 64, NUM_CLASSES).to(DEVICE)
    opt  = torch.optim.Adam(meta.parameters(), lr=0.01)
    bvl, bst, pat = float("inf"), None, 0

    for _ in range(MAX_EPOCHS):
        meta.train()
        opt.zero_grad()
        crit(meta(tr_b), tr_lb).backward()
        opt.step()
        meta.eval()
        with torch.no_grad():
            vl = crit(meta(va_b), va_lb).item()
        if vl < bvl:
            bvl = vl
            bst = copy.deepcopy(meta.state_dict())
            pat = 0
        else:
            pat += 1
            if pat >= META_PATIENCE:
                break

    if bst:
        meta.load_state_dict(bst)
    meta.eval()
    with torch.no_grad():
        out   = meta(te_b)
        probs = torch.softmax(out, 1).cpu().numpy()
        preds = torch.argmax(out, 1).cpu().numpy()
    return np.array(y_te), preds, probs

# ============================================================
# 16) OOF BOOKKEEPING
# ============================================================

def build_oof_rows(ids, y_true, y_pred, y_prob,
                   fold_id, cohort, exp_family,
                   exp_name, setting, model_name):
    return [{
        "cohort":            cohort,
        "experiment_family": exp_family,
        "experiment_name":   exp_name,
        "setting":           setting,
        "model":             model_name,
        "fold":              fold_id,
        "subject_id":        ids[i],
        "y_true":            int(y_true[i]),
        "y_pred":            int(y_pred[i]),
        "prob_class_0":      float(y_prob[i, 0]),
        "prob_class_1":      float(y_prob[i, 1]),
        "prob_class_2":      float(y_prob[i, 2]),
    } for i in range(len(y_true))]


def summarize_oof(oof_df, cohort, exp_family,
                  exp_name, setting, model_name):
    yt = oof_df["y_true"].values.astype(int)
    yp = oof_df["y_pred"].values.astype(int)
    yb = oof_df[["prob_class_0",
                  "prob_class_1",
                  "prob_class_2"]].values.astype(float)
    m  = compute_final_metrics(yt, yp, yb)
    m.update(dict(cohort=cohort, experiment_family=exp_family,
                  experiment_name=exp_name, setting=setting,
                  model=model_name,
                  n_total_oof_samples=len(oof_df)))
    return m

# ============================================================
# 17) CV EXPERIMENT RUNNERS
#
#   Each runner receives `fold_cache` (list of 5 pre-computed dicts)
#   instead of raw arrays.  No imputation happens inside runners.
#   UDS-only runners skip MRI arrays entirely.
# ============================================================

def _log(yt, yp, yb, tag):
    fm = compute_final_metrics(yt, yp, yb)
    print(f"{tag} "
          f"acc={fm['accuracy']:.4f} "
          f"auc_macro={fm['auc_macro_ovr']:.4f} "
          f"auc_bin={fm['auc_binary_collapse_debug']:.4f}")


# ---------- UNIMODAL NON-GRAPH ----------

def run_cv_unimodal_non_graph(model_name, modality_name,
                               fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(100 + fid)
        t0  = time.time()

        if modality_name == "UDS":
            Xtr, Xva, Xte = fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"]
        else:
            Xtr, Xva, Xte = fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"]

        yt, yp, yb = run_non_graph_model(
            model_name, Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "UNIMODAL", "UNIMODAL_NON_GRAPH",
            modality_name, model_name))
        _log(yt, yp, yb,
             f"[UNI][NON_GRAPH][{modality_name}][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "UNIMODAL",
                            "UNIMODAL_NON_GRAPH", modality_name,
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- UNIMODAL GRAPH ----------

def run_cv_unimodal_graph(model_name, modality_name,
                           fold_cache, cv_splits,
                           adjacency, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(200 + fid)
        t0  = time.time()

        if modality_name == "UDS":
            Xtr, Xva, Xte = fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"]
        else:
            Xtr, Xva, Xte = fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"]

        tr_g, va_g, te_g = build_graphs_from_fold(
            Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"], adjacency)
        yt, yp, yb = run_graph_model(
            model_name, tr_g, va_g, te_g,
            in_channels=1, y_tr=fd["y_tr"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "UNIMODAL", "UNIMODAL_GRAPH",
            f"{modality_name}_{structure_name}", model_name))
        _log(yt, yp, yb,
             f"[UNI][GRAPH][{modality_name}][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "UNIMODAL",
                            "UNIMODAL_GRAPH",
                            f"{modality_name}_{structure_name}",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- EARLY FUSION NON-GRAPH ----------

def run_cv_early_fusion_non_graph(model_name, fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(300 + fid)
        t0  = time.time()

        Xtr = np.concatenate([fd["X_uds_tr"], fd["X_mri_tr"]], axis=1)
        Xva = np.concatenate([fd["X_uds_va"], fd["X_mri_va"]], axis=1)
        Xte = np.concatenate([fd["X_uds_te"], fd["X_mri_te"]], axis=1)

        yt, yp, yb = run_non_graph_model(
            model_name, Xtr, Xva, Xte,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "FUSION", "EARLY_FUSION_NON_GRAPH",
            "early", model_name))
        _log(yt, yp, yb,
             f"[EARLY_FUSION][NON_GRAPH][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "EARLY_FUSION_NON_GRAPH", "early",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- LATE FUSION NON-GRAPH ----------

def run_cv_late_fusion_non_graph(model_name, fold_cache, cv_splits):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(400 + fid)
        t0  = time.time()

        _, _, p_uds = run_non_graph_model(
            model_name,
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"])
        _, _, p_mri = run_non_graph_model(
            model_name,
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"])

        prob = 0.5 * (p_uds + p_mri)
        pred = np.argmax(prob, axis=1)

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], fd["y_te"], pred, prob, fid,
            "ALIGNED", "FUSION", "LATE_FUSION_NON_GRAPH",
            "late", model_name))
        _log(fd["y_te"], pred, prob,
             f"[LATE_FUSION][NON_GRAPH][{model_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "LATE_FUSION_NON_GRAPH", "late",
                            model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- EARLY FUSION GRAPH ----------

def run_cv_early_fusion_graph(model_name, fold_cache, cv_splits,
                               uds_adj, mri_adj, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(500 + fid)
        t0  = time.time()

        tr_g, va_g, te_g = build_early_fusion_graphs(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"],
            uds_adj, mri_adj)
        yt, yp, yb = run_graph_model(
            model_name, tr_g, va_g, te_g,
            in_channels=1, y_tr=fd["y_tr"])

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "FUSION", "EARLY_FUSION_GRAPH",
            f"early_{structure_name}", model_name))
        _log(yt, yp, yb,
             f"[EARLY_FUSION][GRAPH][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "EARLY_FUSION_GRAPH",
                            f"early_{structure_name}", model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- LATE FUSION GRAPH ----------

def run_cv_late_fusion_graph(model_name, fold_cache, cv_splits,
                              uds_adj, mri_adj, structure_name):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(600 + fid)
        t0  = time.time()

        tr_ug, va_ug, te_ug = build_graphs_from_fold(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], uds_adj)
        tr_mg, va_mg, te_mg = build_graphs_from_fold(
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], mri_adj)

        _, _, p_uds = run_graph_model(
            model_name, tr_ug, va_ug, te_ug,
            in_channels=1, y_tr=fd["y_tr"])
        _, _, p_mri = run_graph_model(
            model_name, tr_mg, va_mg, te_mg,
            in_channels=1, y_tr=fd["y_tr"])

        prob = 0.5 * (p_uds + p_mri)
        pred = np.argmax(prob, axis=1)

        times.append(time.time() - t0)
        all_rows.extend(build_oof_rows(
            fd["ids_te"], fd["y_te"], pred, prob, fid,
            "ALIGNED", "FUSION", "LATE_FUSION_GRAPH",
            f"late_{structure_name}", model_name))
        _log(fd["y_te"], pred, prob,
             f"[LATE_FUSION][GRAPH][{model_name}]"
             f"[{structure_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    summary = summarize_oof(oof_df, "ALIGNED", "FUSION",
                            "LATE_FUSION_GRAPH",
                            f"late_{structure_name}", model_name)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary


# ---------- PROPOSED FUSION ----------

def _get_proposed_adjs(setting, uds_s, mri_s, fid):
    if setting == "structured":
        return uds_s, mri_s
    if setting == "previous_unstructured":
        return build_identity_adj(uds_s.shape[0]), \
               build_identity_adj(mri_s.shape[0])
    if setting == "new_unstructured_bad":
        return build_bad_unstructured_adj(uds_s, 1000 + fid), \
               build_bad_unstructured_adj(mri_s, 2000 + fid)
    raise ValueError(setting)


def run_cv_proposed_fusion(setting_name,
                            fold_cache, cv_splits,
                            uds_struct_adj, mri_struct_adj):
    all_rows, times = [], []
    for split in cv_splits:
        fid = split["fold"]
        fd  = fold_cache[fid]
        set_seed(700 + fid)
        t0  = time.time()

        uds_adj, mri_adj = _get_proposed_adjs(
            setting_name, uds_struct_adj, mri_struct_adj, fid)

        tr_ug, va_ug, te_ug = build_graphs_from_fold(
            fd["X_uds_tr"], fd["X_uds_va"], fd["X_uds_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], uds_adj)
        tr_mg, va_mg, te_mg = build_graphs_from_fold(
            fd["X_mri_tr"], fd["X_mri_va"], fd["X_mri_te"],
            fd["y_tr"], fd["y_va"], fd["y_te"], mri_adj)

        yt, yp, yb = run_proposed_fusion(
            tr_ug, va_ug, te_ug,
            tr_mg, va_mg, te_mg,
            fd["y_tr"], fd["y_va"], fd["y_te"])

        times.append(time.time() - t0)
        mname = f"ImprovedGTN_MetaGraphClassifier_{setting_name}"
        all_rows.extend(build_oof_rows(
            fd["ids_te"], yt, yp, yb, fid,
            "ALIGNED", "PROPOSED", "PROPOSED_FUSION",
            setting_name, mname))
        _log(yt, yp, yb,
             f"[PROPOSED][{setting_name}][f={fid}]")

    oof_df  = pd.DataFrame(all_rows)
    mname   = f"ImprovedGTN_MetaGraphClassifier_{setting_name}"
    summary = summarize_oof(oof_df, "ALIGNED", "PROPOSED",
                            "PROPOSED_FUSION", setting_name, mname)
    summary["avg_fold_time_seconds"] = float(np.mean(times))
    return oof_df, summary

# ============================================================
# 18) MAIN
# ============================================================

def main():
    all_oof, all_summary = [], []

    # ----------------------------------------------------------
    # LOAD — all 183K+ rows, -999 sentinel intact in MRI
    # ----------------------------------------------------------
    data        = load_v3_data(DATA_DIR)
    y           = data["y"]
    ids         = data["ids"]
    X_uds       = data["uds_arr"]
    X_mri_raw   = data["mri_arr_raw"]
    uds_widths  = data["uds_widths"]
    n_mri_nodes = data["n_mri_nodes"]

    print(f"\n[MAIN] N={len(y)}  "
          f"UDS={X_uds.shape[1]}  MRI={X_mri_raw.shape[1]}\n")

    # ----------------------------------------------------------
    # ADJACENCY
    # ----------------------------------------------------------
    uds_adj_s = build_uds_structured_adj(uds_widths)
    uds_adj_u = build_identity_adj(X_uds.shape[1])
    mri_adj_s = build_mri_structured_adj(n_mri_nodes, MRI_ADJ_PATH)
    mri_adj_u = build_identity_adj(n_mri_nodes)

    # ----------------------------------------------------------
    # CV SPLITS  (full dataset, stratified)
    # ----------------------------------------------------------
    cv_splits = generate_cv_splits(y, n_splits=OUTER_N_SPLITS,
                                   base_seed=123)

    # ----------------------------------------------------------
    # PRE-COMPUTE FOLD ARRAYS  (5 median fits, done once)
    # ----------------------------------------------------------
    print("\n[CACHE] Pre-computing fold arrays (UDS-informed Ridge imputation)...")
    fold_cache = precompute_fold_arrays(
        X_uds, X_mri_raw, y, ids, cv_splits)
    print("[CACHE] Done.\n")

    # ----------------------------------------------------------
    # NON-GRAPH UNIMODAL
    # ----------------------------------------------------------
    non_graph_models = ["LogisticRegression", "MLP",
                        "XGBoost", "CNN", "Transformer"]

    for mn in non_graph_models:
        if mn == "XGBoost" and not XGB_AVAILABLE:
            continue
        for mod in ("UDS", "MRI"):
            oof, s = run_cv_unimodal_non_graph(
                mn, mod, fold_cache, cv_splits)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # GRAPH UNIMODAL
    # ----------------------------------------------------------
    graph_models = ["GCN", "GAT"]

    for gm in graph_models:
        for sname, adj in [("structured",   uds_adj_s),
                            ("unstructured", uds_adj_u)]:
            oof, s = run_cv_unimodal_graph(
                gm, "UDS", fold_cache, cv_splits, adj, sname)
            all_oof.append(oof); all_summary.append(s)

        for sname, adj in [("structured",   mri_adj_s),
                            ("unstructured", mri_adj_u)]:
            oof, s = run_cv_unimodal_graph(
                gm, "MRI", fold_cache, cv_splits, adj, sname)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # NON-GRAPH FUSION
    # ----------------------------------------------------------
    for mn in non_graph_models:
        if mn == "XGBoost" and not XGB_AVAILABLE:
            continue
        oof, s = run_cv_early_fusion_non_graph(
            mn, fold_cache, cv_splits)
        all_oof.append(oof); all_summary.append(s)

        oof, s = run_cv_late_fusion_non_graph(
            mn, fold_cache, cv_splits)
        all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # GRAPH FUSION
    # ----------------------------------------------------------
    for gm in graph_models:
        for sname, ua, ma in [
                ("structured",   uds_adj_s, mri_adj_s),
                ("unstructured", uds_adj_u, mri_adj_u)]:
            oof, s = run_cv_early_fusion_graph(
                gm, fold_cache, cv_splits, ua, ma, sname)
            all_oof.append(oof); all_summary.append(s)

            oof, s = run_cv_late_fusion_graph(
                gm, fold_cache, cv_splits, ua, ma, sname)
            all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # PROPOSED FUSION
    # ----------------------------------------------------------
    for setting in ["structured",
                    "previous_unstructured",
                    "new_unstructured_bad"]:
        oof, s = run_cv_proposed_fusion(
            setting, fold_cache, cv_splits,
            uds_adj_s, mri_adj_s)
        all_oof.append(oof); all_summary.append(s)

    # ----------------------------------------------------------
    # SAVE
    # ----------------------------------------------------------
    oof_all    = pd.concat(all_oof, axis=0, ignore_index=True)
    summary_df = pd.DataFrame(all_summary)

    col_order = [
        "cohort", "experiment_family", "experiment_name", "setting",
        "model", "n_total_oof_samples",
        "accuracy", "auc_macro_ovr", "auc_binary_collapse_debug",
        "sensitivity_macro", "specificity_macro",
        "sensitivity_at_spec80_binary",
        "achieved_specificity_at_spec80_binary",
        "threshold_for_spec80_binary",
        "avg_fold_time_seconds",
    ]
    summary_df = summary_df[col_order]

    print("\n" + "=" * 120)
    print("FINAL OOF SUMMARY")
    print("=" * 120)
    print(summary_df.to_string(index=False))

    oof_all.to_csv("multiclass_5fold_oof_predictions.csv", index=False)
    summary_df.to_csv("multiclass_5fold_final_summary.csv", index=False)
    print("\nSaved: multiclass_5fold_oof_predictions.csv")
    print("       multiclass_5fold_final_summary.csv")


if __name__ == "__main__":
    main()

Using device: cpu
[DATA] total rows           : 185831
[DATA] MRI-missing rows     : 183452  (98.7 %)
[DATA] fully-aligned rows   : 2379
[DATA] UDS dim              : 134  (history=66, survey=39, testing=29)
[DATA] MRI dim              : 198
[DATA] Class distribution   : {0: 97491, 1: 32490, 2: 55850}

[MAIN] N=185831  UDS=134  MRI=198

[ADJ] Identity adjacency for MRI  (n=198)

[CACHE] Pre-computing fold arrays (UDS-informed Ridge imputation)...
[CACHE] fold 0 imputed  train=130081  val=18583  test=37167
[CACHE] fold 1 imputed  train=130081  val=18584  test=37166
[CACHE] fold 2 imputed  train=130081  val=18584  test=37166
[CACHE] fold 3 imputed  train=130081  val=18584  test=37166
[CACHE] fold 4 imputed  train=130081  val=18584  test=37166
[CACHE] Done.

[UNI][NON_GRAPH][UDS][LogisticRegression][f=0] acc=0.7667 auc_macro=0.8942 auc_bin=0.9205
[UNI][NON_GRAPH][UDS][LogisticRegression][f=1] acc=0.7706 auc_macro=0.8965 auc_bin=0.9237
[UNI][NON_GRAPH][UDS][LogisticRegression][f=2] acc=0.7